<a href="https://colab.research.google.com/github/ai-socialimpact/LLMWiki-NGO-Humaninloop/blob/main/Nand_Wiki_creation_v3_1s.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-cloud-aiplatform==1.35.0 google-auth==2.22.0
!pip install --upgrade google-cloud-aiplatform --user


# Section 1: Pydantic definitions

In [ ]:
import os
import json
from typing import List, Optional, Any
from pydantic import BaseModel, Field, ValidationError
from google import genai
from google.genai import types
from google.oauth2 import service_account

# =====================================================================
# 🧬 STEP 1: PYDANTIC DATA STRUCTURAL CONTRACTS & METADATA SCHEMAS
# =====================================================================

class Pass1ETopicBlueprint(BaseModel):
    """
    Defines the structural blueprint for an individual mutually exclusive topic.
    Serves as the definitive data model to drive subsequent ingestion loops.
    """
    topic_slug: str = Field(
        ...,
        description="URL-safe, filesystem-safe lowercase identifier string using underscores with zero extensions  "
    )
    display_name: str = Field(
        ...,
        description="Human-readable title used for the document's primary H1 markdown header  "
    )
    scope_definition: str = Field(
        ...,
        description="Strict, dense definition explicitly outlining what specific operational metrics, guidelines, and rules belong under this topic header."
    )
    exclusion_rules: str = Field(
        ...,
        description="Ironclad rule explicitly detailing what related parameters or background information are strictly FORBIDDEN inside this file container to prevent data leakage across wiki pages."
    )
    contributing_raw_files: List[str] = Field(
        ...,
        description="A list containing the exact, literal string filenames of the original raw markdown source documents that contain data belonging under this specific topic."
    )
    target_seed_keywords: List[str] = Field(
        ...,
        description="A clean list of lowercase, standalone keyword hashtags that encapsulate the unique content boundaries of this topic, used to tag the downstream QA matrix."
    )

class MasterTaxonomyBlueprint(BaseModel):
    """
    The master configuration contract generated at the end of the Phase 1 planning passes.
    Acts as the automated orchestration roadmap for the compilation engine.
    """
    global_reasoning: str = Field(
        ...,
        description="Internal executive thinking log capturing the architectural reasoning and optimization steps used to compress the raw files into balanced, demand-driven themes."
    )
    permitted_topics: List[Pass1ETopicBlueprint] = Field(
        ...,
        description="An exhaustive collection of mutually exclusive, demand-aligned master topic blueprint objects."
    )

class AuditIssue(BaseModel):
    """
    Structures an individual graph validation exception or indexing flaw detected by the system linter.
    """
    issue_id: str = Field(
        ...,
        description="Sequential alphanumeric tracking token identifying the specific audit entry (e.g., AUDIT-01)"
    )
    severity: str = Field(
        ...,
        description="The risk classification token. Enforced to be strictly: CRITICAL, WARNING, or INFO."
    )
    target_file: str = Field(
        ...,
        description="The exact name of the generated index file where the structural error currently resides (index.md or index_detailed.md)"
    )
    description: str = Field(
        ...,
        description="A deeply detailed explanation pinpointing the relative link breakage, orphan file gap, or semantic context omission."
    )

class VerificationReport(BaseModel):
    """
    The final schema contract mapping total graph network validation integrity.
    Determines if the vault matches ground-truth folder states cleanly.
    """
    is_vault_structurally_sound: bool = Field(
        ...,
        description="Set to true ONLY if zero errors, broken paths, typos, or dropped categories are detected by the auditor."
    )
    detected_anomalies: List[AuditIssue] = Field(
        ...,
        description="A comprehensive array listing every flagged indexing gap or broken relative link found across the index layers."
    )


# Section Inputs: keys

#Section 2

In [ ]:
# =====================================================================
# 🛰️ STEP 2: ENTERPRISE VERTEX AI SECURE RUNNER ENGINE (generateNEWNEWNEW)
# =====================================================================

# @title
from google import genai
from google.genai import types
import base64
from google.oauth2 import service_account

def getTheCredentials():
    credentials = service_account.Credentials.from_service_account_file(
         "/content/vertexaikey2.json",
        scopes=["https://www.googleapis.com/auth/cloud-platform"],
    )

    return credentials


# Instantiating the global client explicitly bound to private Vertex AI infrastructure.
# Injects the loaded service account credentials natively.
client = genai.Client(
    vertexai=True,
    project=GCP_PROJECT_ID,
    location=GCP_LOCATION,
    credentials=getTheCredentials()
)

client_instance = client



In [ ]:
# =====================================================================
# 🛰️ RESILIENT ENGINE LAYER: execute_gemini_call & generateNEWNEWNEW
# =====================================================================

import random
import time

def execute_gemini_call(
    client_instance,
    custompromptstring,
    myparts,
    mymodel,
    mytemperature,
    my_max_output_tokens,
    expected_response_schema,
    my_system_instruction,
    my_tools
):
    """
    Inner Worker Function: Executes the low-level API call to Vertex AI.
    Contains zero retry logic. Handled inside standard SDK configurations.
    """
    text1 = types.Part.from_text(text=custompromptstring)
    partstoSendtoGemini = [text1]

    for aitem in myparts:
        if aitem is not None:
            partstoSendtoGemini.append(aitem)

    contents = [
        types.Content(role="user", parts=partstoSendtoGemini)
    ]

    system_parts = None
    if my_system_instruction is not None:
        system_parts = [types.Part.from_text(text=my_system_instruction)]
    else:
        system_parts = [types.Part.from_text(text="You are a helpful assistant.")]

    response_mime = "application/json" if expected_response_schema else "text/plain"

    generate_content_config = types.GenerateContentConfig(
        temperature=mytemperature,
        top_p=0.95,
        max_output_tokens=my_max_output_tokens,
        response_modalities=["TEXT"],
        response_mime_type=response_mime,
        response_schema=expected_response_schema,
        system_instruction=system_parts,
        tools=my_tools,
        safety_settings=[
            types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF")
        ]
    )

    return client_instance.models.generate_content(
        model=mymodel,
        contents=contents,
        config=generate_content_config,
    )


def generateNEWNEWNEW(
    client_instance,
    custompromptstring=None,
    myparts=[],
    mymodel="gemini-2.5-flash",
    mytemperature=0,
    my_max_output_tokens=24000,
    expected_response_schema=None,
    my_system_instruction=None,
    my_tools=None,
    max_retries=5,
    initial_backoff_base=2.0
):
    """
    Resilient Wrapper Layer: Retains your exact signature footprint.
    Intercepts 429 RESOURCE_EXHAUSTED responses and automatically executes
    truncated exponential backoff with randomized jitter to safeguard loop execution.
    """
    attempt = 0

    while True:
        try:
            # Dispatch directly down to our renamed inner text execution worker
            return execute_gemini_call(
                client_instance=client_instance,
                custompromptstring=custompromptstring,
                myparts=myparts,
                mymodel=mymodel,
                mytemperature=mytemperature,
                my_max_output_tokens=my_max_output_tokens,
                expected_response_schema=expected_response_schema,
                my_system_instruction=my_system_instruction,
                my_tools=my_tools
            )

        except Exception as api_exception:
            exception_message = str(api_exception)

            # Catch both explicit status strings or numeric error code blocks from the API
            if "429" in exception_message or "RESOURCE_EXHAUSTED" in exception_message:
                attempt += 1

                # Escalation fallback gate if quota wall persists indefinitely
                if attempt > max_retries:
                    print(f"\n💥 [Resilient Shield]: Maximum retry quota ({max_retries}) breached. Escalating error context.")
                    raise api_exception

                # Calculate Truncated Exponential Delay + Randomized Jitter offset to prevent collision overlaps
                sleep_duration = (initial_backoff_base ** attempt) + random.uniform(0.1, 1.5)

                print(f"⚠️ [API Alert 429]: Quota Resource Exhausted detected during loop transaction.")
                print(f"    Attempt [{attempt}/{max_retries}]: Sleep backing off for {sleep_duration:.2f} seconds before retry...")

                time.sleep(sleep_duration)
            else:
                # Instantly escalate typical compilation exceptions or standard programming syntax bugs
                raise api_exception


In [ ]:
client

# Section 3: File handling util functions

In [ ]:
# =====================================================================
# 📁 STEP 3: REUSABLE HARD DISK & TARGETED INGESTION I/O FILE HELPERS
# =====================================================================

def list_md(directory: str) -> List[str]:
    """
    Scans a targeted filesystem directory and filters exclusively for
    markdown filenames. Automatically strips away hidden system folders.
    """
    if not os.path.exists(directory):
        return []
    # Harvests standard .md extensions while ignoring temporary Colab notebook artifacts
    return [
        f for f in os.listdir(directory)
        if f.lower().endswith('.md') and not f.startswith('.')
    ]

def read_md(file_path: str) -> str:
    """
    Safely reads the text content from an individual markdown file
    using strict UTF-8 string encoding formats.
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()

def write_md(file_path: str, content: str) -> None:
    """
    Writes a plain text string directly to the filesystem. Programmatically
    creates missing parent folders if they do not exist on disk.
    """
    parent_directory = os.path.dirname(file_path)
    if parent_directory:
        os.makedirs(parent_directory, exist_ok=True)
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(content)

def wrap_file_with_boundary(directory: str, filename: str) -> types.Part:
    """
    Loads raw text data from a specific file path and structures it into
    a bounded, labelled context Part object for Gemini ingestion.
    """
    clean_filename = filename.strip()
    full_target_path = os.path.join(directory, clean_filename)

    # Extract raw content strings directly
    raw_markdown_text = read_md(full_target_path)

    # Construct an ironclad visual template layout boundary block
    wrapped_context_string = (
        f"--- START FILE: {clean_filename} ---\n"
        f"{raw_markdown_text}\n"
        f"--- END FILE: {clean_filename} ---\n"
    )

    # Package natively as an SDK text Part block
    return types.Part.from_text(text=wrapped_context_string)

def package_targeted_sources(directory: str, required_filenames: List[str]) -> List[types.Part]:
    """
    Acts as the targeted ingestion gatekeeper for execution loops. Scans disk
    and packages ONLY the explicitly requested source files for a given topic container.
    """
    packaged_context_parts = []

    for current_filename in required_filenames:
        clean_name = current_filename.strip()
        full_disk_path = os.path.join(directory, clean_name)

        # Hard check to protect execution stability inside the Colab notebook cells
        if not os.path.exists(full_disk_path):
            print(f"⚠️ Ingestion Warning: Mapped document tracking reference '{clean_name}' "
                  f"is completely missing from physical path '{directory}'. Skipping element.")
            continue

        # Call our standard wrapper block to append the bounded context part
        bounded_part = wrap_file_with_boundary(directory, clean_name)
        packaged_context_parts.append(bounded_part)

    return packaged_context_parts

def parse_response_to_dict(response_obj: Any, validation_schema: Any) -> dict:
    """
    Extracts structured JSON dictionary objects from model responses.
    Prioritizes native SDK .parsed fields with a fallback backtick string cleaner.
    """
    # Standard high-integrity structural data check
    if response_obj.parsed is not None:
        return response_obj.parsed.model_dump()

    print("⚠️ SDK native parsing bypassed. Activating structural markdown block regex cleanup...")
    raw_response_text = response_obj.text.strip()
    cleaned_json_string = raw_response_text

    # Clean out markdown codeblock formatting if returned by text channels
    if "```json" in cleaned_json_string:
        cleaned_json_string = cleaned_json_string.split("```json")[-1].split("```")[0].strip()
    elif "```" in cleaned_json_string:
        cleaned_json_string = cleaned_json_string.split("```")[-1].split("```")[0].strip()

    try:
        # Enforce runtime Pydantic schema validation checks before JSON evaluation
        validation_schema.model_validate_json(cleaned_json_string)
    except ValidationError as structure_fault:
        print("\n💥 =================== PYDANTIC INTEGRITY REJECTION ===================")
        print(structure_fault)
        print("=======================================================================\n")

    return json.loads(cleaned_json_string)


# Step 1 :  Run Pass 1. Pass 1 proposes a list of topics

Run all pass1A,pass1B,Pass1C,Pass1D,Pass1E


*   Mandatoary Input: The source folder name NGO_SOURCE_DOCS with all files
*   Optional Input 1: A guidance note based on use case. Currently describes SNEHA use case
*   Optional Input 2: A log of all questions asked . Currently is Empty (default).



## Pass 1 inputs

In [ ]:
# Define standard target workspace paths matching storage footprints
SOURCE_RAW_DIR = "/content/NGO_SOURCE_DOCS/"
OUTPUT_PLAN_DIR = "/content/Pass1Output/"
COMMUNITY_LOG_FILE = "/content/logs/community_queries.txt"


In [ ]:
# =====================================================================
# THE SPECIALISED NGO WHATSAPP CHATBOT TAXONOMY TEMPLATE
# =====================================================================

# This is Optional .. (Can be an empty string too)

WikiTEMPLATE_Guidance = """

CRITICAL BUSINESS USE CASE CONTEXT & THEMATIC BOUNDARY LOCKS:
The downstream application is a Question Answering WhatsApp Chatbot designed provide answers to questions from the community (usually endusers of the WhatsApp Chatbot are from low litracy or rural areas)
Users naturally frame questions on the Wiki contents, as that is relavent for them.

The downstream module is a Question Answering system that answers to questions from public endusers based on the Wiki contents.
The downstream module uses a 2 pass LLM Wiki architecure pattern (Karpathy's LLM Wiki) consisting Router LLM and a Question Answer Extraction LLM.
In this 2 pass Question Answering system, the Router LLM (gpt4o) identifis the topic(s) of the question, and suggests the identified topic files to look at from the Wiki folder.
Based on the identified files, the Question Answer Extraction LLM reads these identified files to find the answer to the user question.
So it is important to organize the LLM Wiki topic taxonomy in a way that is effective for this downstream question answering architecture.

To maximize downstream question-answering routing utility and achieve perfect isolation for GPT-4o attention layers,
you must actively organize these topic boundaries strictly into a balanced directory taxonomy of distinct, mutually exclusive topic containers.

Identify the domain of the use case based on the input files and what the likely user community is likely to ask.
Use your guess of the identified domain to reason out the possible taxonomy and number of topics required for the LLM Wiki (Karpathy).

For the 2 pass Question Answering system of the LLM Wiki to be effective, determine the total number of topic containers based the identified domain/use case, and structural diversity/size of the raw input files.
The final node count must be driven purely by requirements of LLM Wiki that is optimized for : every container must be completely mutually exclusive from its neighbors while preserving a dense, actionable core of facts.

Here are some possible ways of organizing the topics in the input raw files into the target LLM Wiki.
You identify how to organize the taxonomy of topic.
You may choose your own layout. Here is a example of topic layout.
TOPIC LAYOUT TEMPLATES:
  1. SPECIALIZED SYSTEMS & COMPONENT ROLES: Group baseline concepts, components, guidelines, and routine operational rules explicitly by their distinct structural specialization or area of expertise.
  2. LIFECYCLE & STAGE HORIZONS: Categorize processes, execution lifecycles, milestone timelines, or structural version brackets into dedicated sequential tracking buckets.
  3. EXCEPTION TRAGETS & TRIGGER CRITICALITY: Isolate severe anomalies, high-risk failures, edge cases, critical emergencies, or escalation workflows into isolated, high-density files so that urgent inquiries route directly.
  4. ADMINISTRATIVE, STATUTORY, & POLICY FRAMEWORKS: Isolate compliance constraints, regulatory rules, prerequisites, qualification frameworks, or system requirements into clear verification paths.
  5. END-USER ADVICE FOR VARIOUS SCENARIOS: Suggestions to community for various type of scenarios/conditions/situation




"""

## Pass 1  (A & B)

In [ ]:
# =====================================================================
# 🏙️ STEP 4 PROMPT STRINGS & SYSTEM INSTRUCTIONS (GLOBAL TEXT CONSTANTS)
# =====================================================================

PROMPT_1A = (
    "Analyze all enclosed markdown documents globally. Your goal is to brainstorm an exhaustive, "
    "comprehensive list of every single possible operational theme, field protocol topic, "
    "keyword boundary, checklist framework, or metric category hidden in the text.\n\n"
    "Actively search for, isolate, and list specific parameters,  guidelines, "
    "numbers, ratios, transport thresholds, timeline constraints, and hidden exceptions.\n\n"
    "Output your raw findings as an unrefined, highly detailed bulleted inventory string. "
    "Do not attempt to summarize, group, truncate, or compress concepts yet. Prioritize maximum recall."
)

SYSTEM_INSTRUCTION_1A = (
    "You are an investigative data architect tracking unstructured field documentation.  "
    "Your role is an unguided scout; your failure "
    "condition is omission. Write extremely dense, fact-based bullet points with zero narrative filler."
)



# =====================================================================
# 🏗️ STEP 4 PROMPT RECALIBRATION: TAXONOMY TARGET TUNING (10 TO 25 TOPICS)
# =====================================================================

TEMPLATE_1B = """Review the attached raw markdown files along with the unstructured concept inventory provided below:

=== RAW CONCEPT DISCOVERY INVENTORY ===
{brainstorm_data}
=======================================

Your core task is to organize the input files into a clean, highly balanced directory taxonomy.

Avoid generic, document-centric titles (like 'appendix_a' or 'general_field_logistics').
Every topic container must represent a distinct user intent path.
Output a structured, plain-text directory layout detailing the proposed master topic slug names, their display titles, and their explicit target timelines.
Do not write JSON code blocks yet.
"""

# Append your updated, context-rich tri-thematic guidance matrix
TEMPLATE_1B = TEMPLATE_1B + WikiTEMPLATE_Guidance

SYSTEM_INSTRUCTION_1B = (
    "You are a strategic taxonomy engineer organizing knowledge architectures for high-speed retrieval engines. "
    "Your objective is to ensure absolute, granular thematic separation between document nodes to support strict "
    "downstream logical routing. Focus on splitting nodes cleanly by your proposed topic taxonomy layout structure (e.g milestone/severity/topic) "
    "Be analytical, logical, and highly structured in your output."
)

print("✅ [System Patch]: TEMPLATE_1B   and SYSTEM_INSTRUCTION_1B updated successfully.")


# =====================================================================
# 🏙️ STEP 4 UTILITY TEMPLATE BLOCK
# =====================================================================

def get_pass_1b_prompt(raw_brainstorm_text: str) -> str:
    """Pure string formatter that injects brainstorm data into the global constant template text."""
    return TEMPLATE_1B.format(brainstorm_data=raw_brainstorm_text)


# =====================================================================
# 🏙️ STEP 4 EXECUTION BLOCK: PASS 1A & 1B METHOD LABELS
# =====================================================================

def execute_pass_1a_global_discovery(all_input_parts: List[types.Part]) -> str:
    """
    Executes a high-recall, global sweep across all source markdown documents.
    Extracts every rule, metric, checklist, and parameter hidden in the text.
    """
    print("\n[Pass 1A] 🔄 Commencing Broad Semantic Concept Discovery Pass...")

    # Standardize on Gemini 2.5 Pro for massive cross-document context evaluation
    response_1a = generateNEWNEWNEW(
        client_instance=client,
        custompromptstring=PROMPT_1A,
        myparts=all_input_parts,
        mymodel="gemini-2.5-pro",
        mytemperature=0,
        my_max_output_tokens=24000,
        expected_response_schema=None,
        my_system_instruction=SYSTEM_INSTRUCTION_1A,
        my_tools=None
    )

    print("[Pass 1A] ✅ Global concept discovery inventory completed.")
    return response_1a.text


def execute_pass_1b_taxonomy_design(all_input_parts: List[types.Part], raw_brainstorm_text: str) -> str:
    """
    Evaluates raw brainstorm concepts against text data volumes across all 14 files.
    Proposes an initial balanced directory taxonomy of 5 to 10 distinct, master topics.
    """
    print("[Pass 1B] 🔄 Commencing Theoretical Directory Topology Design...")

    # Format prompt text through the global text constant container
    prompt_1b = get_pass_1b_prompt(raw_brainstorm_text)

    response_1b = generateNEWNEWNEW(
        client_instance=client,
        custompromptstring=prompt_1b,
        myparts=all_input_parts,
        mymodel="gemini-2.5-pro",
        mytemperature=0,
        my_max_output_tokens=24000,
        expected_response_schema=None,
        my_system_instruction=SYSTEM_INSTRUCTION_1B,
        my_tools=None
    )

    print("[Pass 1B] ✅ Theoretical taxonomy directory framework generated.")
    return response_1b.text


✅ [System Patch]: TEMPLATE_1B   and SYSTEM_INSTRUCTION_1B updated successfully.


In [ ]:
print(TEMPLATE_1B)

Review the attached raw markdown files along with the unstructured concept inventory provided below:

=== RAW CONCEPT DISCOVERY INVENTORY ===
{brainstorm_data}

Your core task is to organize the input files into a clean, highly balanced directory taxonomy.

Avoid generic, document-centric titles (like 'appendix_a' or 'general_field_logistics').
Every topic container must represent a distinct user intent path.
Output a structured, plain-text directory layout detailing the proposed master topic slug names, their display titles, and their explicit target timelines.
Do not write JSON code blocks yet.


CRITICAL BUSINESS USE CASE CONTEXT & THEMATIC BOUNDARY LOCKS:
The downstream application is a Question Answering WhatsApp Chatbot designed provide answers to questions from the community (usually endusers of the WhatsApp Chatbot are from low litracy or rural areas)
Users naturally frame questions on the Wiki contents, as that is relavent for them.

The downstream module is a Question Answeri

## Pass 1 (C & D)

In [ ]:
# =====================================================================
# 🏙️ STEP 5 PROMPT STRINGS & SYSTEM INSTRUCTIONS (GLOBAL TEXT CONSTANTS)
# =====================================================================

TEMPLATE_1C = (
    "You are a Demand-Driven Knowledge Architect. Your single objective is to optimize a proposed topic "
    "directory layout to maximize downstream query routing accuracy.\n\n"
    "Here is the proposed theoretical master topic list generated in the previous stage:\n"
    "=== THEORETICAL TOPIC LIST ===\n"
    "{topic_data}\n"
    "==============================\n\n"
    "Here is the historical Community QA Query Log data harvested from field deployment:\n"
    "=== HISTORICAL COMMUNITY QA LOG ===\n"
    "{log_data}\n"
    "===================================\n\n"
    "CRITICAL CONDITIONAL EXECUTION INSTRUCTIONS:\n"
    "- CASE 1: If the Community QA Log contains the literal token 'COLD_START_EMPTY_LOG', this indicates "
    "a Month 1 system deployment. Review the proposed list for logical clarity. Do not warp, split, "
    "or structurally alter the taxonomy definitions. Pass the baseline theoretical list straight through "
    "as your final output.\n"
    "- CASE 2: If the log contains real, conversational user questions, this indicates Year 1+ system maturity. "
    "Analyze the public query distribution carefully. Split, merge, reshape, or warp the topic boundaries "
    "so that the final page layout perfectly matches the frequency of questions actually asked by the community. "
    "High-traffic queries must yield granular pages; low-traffic questions must stay compressed in broad pages.\n\n"
    "Output your refined, optimized topic matrix layout string as clean plain text. Do not write JSON or markdown codeblocks."
)

SYSTEM_INSTRUCTION_1C = (
    "You are an adaptive systems architect mapping knowledge category models to real-world user intent "
    "distribution logs. Your goal is alignment, balance, and real-world retrieval utility."
)

TEMPLATE_1D = (
    "Review the following demand-optimized topic matrix layout:\n\n"
    "=== REFINED TOPIC MATRIX ===\n"
    "{aligned_data}\n"
    "=============================\n\n"
    "Your task is to act as an editorial board to polish the definitions and harden the boundaries for each topic node.\n\n"
    "For every single topic container listed, you must write:\n"
    "1. A precise narrative definition detailing exactly what specific guidelines and metrics belong inside its scope.\n"
    "2. A strict EXCLUSION RULE detailing exactly what related background parameters or rules are explicitly "
    "FORBIDDEN from entering this file container.\n\n"
    "By drawing these defensive lines in the sand, you guarantee that information is never scattered or duplicated "
    "across the vault. Output your polished taxonomy structure with ironclad rules as structured plain text."
)

SYSTEM_INSTRUCTION_1D = (
    "You are an ironclad enterprise copyeditor enforcing strict containment boundaries between document scopes "
    "to completely eliminate data leakage. Be absolute, authoritative, and direct in your rules."
)

TEMPLATE_1C = TEMPLATE_1C + WikiTEMPLATE_Guidance

TEMPLATE_1D = TEMPLATE_1D + WikiTEMPLATE_Guidance


# =====================================================================
# 🏙️ STEP 5 UTILITY TEMPLATE BLOCKS
# =====================================================================

def get_pass_1c_prompt(theoretical_topics_text: str, community_log_text: str) -> str:
    """Injects theoretical topic strings and log datasets into the Pass 1C execution prompt template."""
    return TEMPLATE_1C.format(topic_data=theoretical_topics_text, log_data=community_log_text)

def get_pass_1d_prompt(aligned_topics_text: str) -> str:
    """Injects aligned taxonomy strings into the Pass 1D boundary hardening prompt template."""
    return TEMPLATE_1D.format(aligned_data=aligned_topics_text)


# =====================================================================
# 🏙️ STEP 5 EXECUTION BLOCK: PASS 1C & 1D METHOD LABELS
# =====================================================================

def execute_pass_1c_query_alignment(theoretical_topics_text: str, community_log_text: str) -> str:
    """
    Evaluates proposed topic containers against real-world community query history log trends.
    Reshapes, splits, or passes the taxonomy based on active ingestion lifecycle states.
    """
    print("[Pass 1C] 🔄 Commencing Demand-Driven Query Alignment Pass...")

    # Format prompt text through the global text constant container
    prompt_1c = get_pass_1c_prompt(theoretical_topics_text, community_log_text)

    # Executed via Gemini 2.5 Pro to manage complex intent re-balancing reasoning
    response_1C = generateNEWNEWNEW(
        client_instance=client,
        custompromptstring=prompt_1c,
        myparts=[],  # Hides raw files to prevent context fragmentation
        mymodel="gemini-2.5-pro",
        mytemperature=0,
        my_max_output_tokens=24000,
        expected_response_schema=None,
        my_system_instruction=SYSTEM_INSTRUCTION_1C,
        my_tools=None
    )

    print("[Pass 1C] ✅ Demand-aligned taxonomy matrix completed.")
    return response_1C.text


def execute_pass_1d_scope_polish(aligned_topics_text: str) -> str:
    """
    Hardens individual topic definitions by creating strict narrative boundaries
    and explicit exclusion rules to prevent info duplication.
    """
    print("[Pass 1D] 🔄 Commencing Scope Polish & Boundary Hardening Pass...")

    # Format prompt text through the global text constant container
    prompt_1d = get_pass_1d_prompt(aligned_topics_text)

    response_1d = generateNEWNEWNEW(
        client_instance=client,
        custompromptstring=prompt_1d,
        myparts=[],  # Pure text reasoning loop
        mymodel="gemini-2.5-pro",
        mytemperature=0,
        my_max_output_tokens=24000,
        expected_response_schema=None,
        my_system_instruction=SYSTEM_INSTRUCTION_1D,
        my_tools=None
    )

    print("[Pass 1D] ✅ Scope definitions and exclusion rules successfully anchored.")
    return response_1d.text


In [ ]:
print(TEMPLATE_1D)

Review the following demand-optimized topic matrix layout:

=== REFINED TOPIC MATRIX ===
{aligned_data}

Your task is to act as an editorial board to polish the definitions and harden the boundaries for each topic node.

For every single topic container listed, you must write:
1. A precise narrative definition detailing exactly what specific guidelines and metrics belong inside its scope.
2. A strict EXCLUSION RULE detailing exactly what related background parameters or rules are explicitly FORBIDDEN from entering this file container.

By drawing these defensive lines in the sand, you guarantee that information is never scattered or duplicated across the vault. Output your polished taxonomy structure with ironclad rules as structured plain text.

CRITICAL BUSINESS USE CASE CONTEXT & THEMATIC BOUNDARY LOCKS:
The downstream application is a Question Answering WhatsApp Chatbot designed provide answers to questions from the community (usually endusers of the WhatsApp Chatbot are from low l

## Pass 1 (E)

In [ ]:
# =====================================================================
# 🏙️ STEP 6 PROMPT STRINGS & SYSTEM INSTRUCTIONS (GLOBAL TEXT CONSTANTS)
# =====================================================================

TEMPLATE_1E = (
    "Review the finalized topic specifications, scopes, and boundaries provided below:\n\n"
    "=== POLISHED TAXONOMY DIRECTORY ===\n"
    "{polished_data}\n"
    "====================================\n\n"
    "Your final task in this planning phase is to act as a matrix compilation gate.\n"
    "Cross-reference these polished topic definitions directly with the attached raw source files. "
    "For each specific topic container:\n"
    "1. Complete the 'contributing_raw_files' array by mapping the exact, literal filenames from the pool "
    "that contain information belonging under its scope.\n"
    "2. Extract a list of clean, lowercase alphanumeric standalone hashtags to populate the "
    "'target_seed_keywords' array.\n\n"
    "Output your response strictly within the requested JSON schema contract format to guarantee "
    "downstream pipeline stability."
)

SYSTEM_INSTRUCTION_1E = (
    "You are an elite data compiler locking complex relational matrix mappings into verified, "
    "deterministic Pydantic contracts. Ensure absolute filename accuracy matching the input files."
)


# =====================================================================
# 🏙️ STEP 6 UTILITY TEMPLATE BLOCK
# =====================================================================

def get_pass_1e_prompt(polished_taxonomy_text: str) -> str:
    """Injects polished taxonomy strings into the Pass 1E matrix mapping template."""
    return TEMPLATE_1E.format(polished_data=polished_taxonomy_text)


# =====================================================================
# 🏙️ STEP 6 EXECUTION BLOCK: PASS 1E METHOD LABEL
# =====================================================================

def execute_pass_1e_matrix_mapping(all_input_parts: List[types.Part], polished_taxonomy_text: str) -> dict:
    """
    Cross-references finalized, polished taxonomy definitions against the 14 raw files.
    Returns a schema-validated dictionary mapping filenames and keywords to topic blocks.
    """
    print("[Pass 1E] 🔄 Commencing Ground-Truth File Matrix Mapping Gate...")

    # Format prompt text through the global text constant container
    prompt_1e = get_pass_1e_prompt(polished_taxonomy_text)

    # Executes via Gemini 2.5 Pro to manage complex schema-binding layout lookups
    response_1e = generateNEWNEWNEW(
        client_instance=client,
        custompromptstring=prompt_1e,
        myparts=all_input_parts,  # Re-injects all 14 files for precise text mapping
        mymodel="gemini-2.5-pro",
        mytemperature=0,
        my_max_output_tokens=24000,
        expected_response_schema=MasterTaxonomyBlueprint,  # Enforces Pydantic structural contract
        my_system_instruction=SYSTEM_INSTRUCTION_1E,
        my_tools=None
    )

    # Pass response object to our multi-tier parse helper from Step 3
    taxonomy_dict = parse_response_to_dict(response_1e, MasterTaxonomyBlueprint)
    print("[Pass 1E] ✅ Master data blueprint matrix locked and validated.")
    return taxonomy_dict


In [ ]:
print(TEMPLATE_1D)

Review the following demand-optimized topic matrix layout:

=== REFINED TOPIC MATRIX ===
{aligned_data}

Your task is to act as an editorial board to polish the definitions and harden the boundaries for each topic node.

For every single topic container listed, you must write:
1. A precise narrative definition detailing exactly what specific guidelines and metrics belong inside its scope.
2. A strict EXCLUSION RULE detailing exactly what related background parameters or rules are explicitly FORBIDDEN from entering this file container.

By drawing these defensive lines in the sand, you guarantee that information is never scattered or duplicated across the vault. Output your polished taxonomy structure with ironclad rules as structured plain text.

CRITICAL BUSINESS USE CASE CONTEXT & THEMATIC BOUNDARY LOCKS:
The downstream application is a Question Answering WhatsApp Chatbot designed provide answers to questions from the community (usually endusers of the WhatsApp Chatbot are from low l

## Exectue Pass 1

In [ ]:
import time

# =====================================================================
# 🏙️ STEP 6.1 EXECUTION BLOCK: COMPREHENSIVE PASS 1 ORCHESTRATION PIPELINE
# =====================================================================

def Pass1_ComprehensiveRun(raw_dir: str, output_planning_dir: str, community_log_path: str) -> Optional[dict]:
    """
    Executes the entire 5-stage Phase 1 planning matrix sequentially.
    Enforces cooling delays for rate limit protection and logs step states directly.
    Saves text blueprints and final JSON contracts to a dedicated folder.
    """
    print("🎬 Initializing [Pass1_ComprehensiveRun] Ingestion Sequence...")
    os.makedirs(output_planning_dir, exist_ok=True)

    # 1. Harvest ground-truth raw source files from disk
    raw_filenames = list_md(raw_dir)
    if not raw_filenames:
        print(f"❌ Pass 1 Execution Fault: No raw source markdown assets (.md) located inside '{raw_dir}'.")
        print("   Please upload your 14 files to the directory before running.")
        return None

    print(f"📦 Discovered {len(raw_filenames)} static source files. Packaging into context parts...")
    all_input_parts = [wrap_file_with_boundary(raw_dir, f) for f in raw_filenames]

    # 2. Establish the Community Log text block baseline state condition
    community_log_text = "COLD_START_EMPTY_LOG"
    if os.path.exists(community_log_path) and os.path.getsize(community_log_path) > 10:
        try:
            community_log_text = read_md(community_log_path)
            print("📑 Populated Community Query Log verified. Activating demand-driven optimization path...")
        except Exception as read_err:
            print(f"⚠️ Non-critical warning reading community log: {str(read_err)}. Falling back to cold-start.")
    else:
        print("❄️ Community Query Log empty/missing. Processing Month 1 Cold-Start Baseline path...")

    # Tracking variables across execution states
    pass_1a_out = None
    pass_1b_out = None
    pass_1c_out = None
    pass_1d_out = None
    final_taxonomy_json_dict = None

    # -----------------------------------------------------------------
    # RUN PASS 1A: GLOBAL DISCOVERY
    # -----------------------------------------------------------------
    try:
        pass_1a_out = execute_pass_1a_global_discovery(all_input_parts)
        write_md(os.path.join(output_planning_dir, "pass_1a_discovery.txt"), pass_1a_out)
        print("📊 [Pass 1A Status]: SUCCESS. Raw blueprint saved to output folder.")
    except Exception as err_1a:
        print(f"💥 [Pass 1A Status]: FAILURE. Critical execution termination. Reason: {str(err_1a)}")
        return None

    # Defensive Cooling Interval (Allows for safe Gemini rate limit processing runways)
    print("⏳ Applying 5-second rate-limiting cooling delay...")
    time.sleep(15)

    # -----------------------------------------------------------------
    # RUN PASS 1B: THEORETICAL TAXONOMY DESIGN
    # -----------------------------------------------------------------
    try:
        pass_1b_out = execute_pass_1b_taxonomy_design(all_input_parts, pass_1a_out)
        write_md(os.path.join(output_planning_dir, "pass_1b_taxonomy.txt"), pass_1b_out)
        print("📊 [Pass 1B Status]: SUCCESS. Theoretical category layout saved to output folder.")
    except Exception as err_1b:
        print(f"💥 [Pass 1B Status]: FAILURE. Critical execution termination. Reason: {str(err_1b)}")
        return None

    print("⏳ Applying 5-second rate-limiting cooling delay...")
    time.sleep(15)

    # -----------------------------------------------------------------
    # RUN PASS 1C: DOWNSTREAM QUERY ALIGNMENT & REFINE
    # -----------------------------------------------------------------
    try:
        pass_1c_out = execute_pass_1c_query_alignment(pass_1b_out, community_log_text)
        write_md(os.path.join(output_planning_dir, "pass_1c_aligned_matrix.txt"), pass_1c_out)
        print("📊 [Pass 1C Status]: SUCCESS. Demand-optimized matrix saved to output folder.")
    except Exception as err_1c:
        print(f"💥 [Pass 1C Status]: FAILURE. Critical execution termination. Reason: {str(err_1c)}")
        return None

    print("⏳ Applying 5-second rate-limiting cooling delay...")
    time.sleep(20)

    # -----------------------------------------------------------------
    # RUN PASS 1D: SCOPE BOUNDARY POLISH & EXCLUSIONS
    # -----------------------------------------------------------------
    try:
        pass_1d_out = execute_pass_1d_scope_polish(pass_1c_out)
        write_md(os.path.join(output_planning_dir, "pass_1d_hardened_boundaries.txt"), pass_1d_out)
        print("📊 [Pass 1D Status]: SUCCESS. Hardened boundaries and exclusion rules saved to output folder.")
    except Exception as err_1d:
        print(f"💥 [Pass 1D Status]: FAILURE. Critical execution termination. Reason: {str(err_1d)}")
        return None

    print("⏳ Applying 5-second rate-limiting cooling delay...")
    time.sleep(25)

    # -----------------------------------------------------------------
    # RUN PASS 1E: GROUND-TRUTH PYDANTIC MATRIX MAPPING
    # -----------------------------------------------------------------
    try:
        final_taxonomy_json_dict = execute_pass_1e_matrix_mapping(all_input_parts, pass_1d_out)

        # Save validated dictionary blueprint object as our master deployment config asset
        json_output_path = os.path.join(output_planning_dir, "master_taxonomy_blueprint.json")
        with open(json_output_path, "w", encoding="utf-8") as json_file:
            json.dump(final_taxonomy_json_dict, json_file, indent=4, ensure_ascii=False)

        print("📊 [Pass 1E Status]: SUCCESS. Schema contract verified and saved to disk.")
        print(f"\n🎉 [Phase 1 Pipeline Completion]: Master Planning Sequence completed with absolute integrity!")
        print(f"   📁 All output design blueprints are stored safely in: {output_planning_dir}")
        return final_taxonomy_json_dict

    except Exception as err_1e:
        print(f"💥 [Pass 1E Status]: FAILURE. Critical structural mapping termination. Reason: {str(err_1e)}")
        return None

# =====================================================================
# 🧪 RUNTIME NOTEBOOK EXECUTOR CELL
# =====================================================================


# Execution trigger wrapper block
try:
    # Executes the comprehensive Pass 1 pipeline
    # NOTE: Ensure you replace your GCP project ID settings in the Step 2 cell before triggering!
    validated_taxonomy_roadmap = Pass1_ComprehensiveRun(
        raw_dir=SOURCE_RAW_DIR,
        output_planning_dir=OUTPUT_PLAN_DIR,
        community_log_path=COMMUNITY_LOG_FILE
    )

except Exception as global_fault:
    print(f"\n❌ Execution pipeline halted by a global system exception: {str(global_fault)}")


🎬 Initializing [Pass1_ComprehensiveRun] Ingestion Sequence...
❌ Pass 1 Execution Fault: No raw source markdown assets (.md) located inside '/content/NGO_SOURCE_DOCS/'.
   Please upload your 14 files to the directory before running.


In [ ]:
#generateNEWNEWNEW(client, custompromptstring='color of sky' )

In [ ]:
import shutil
import os

folder_to_delete = '/content/LLM_Wiki_VaultNONME'

if os.path.exists(folder_to_delete):
    shutil.rmtree(folder_to_delete)
    print(f"Folder '{folder_to_delete}' and its contents deleted successfully.")
else:
    print(f"Folder '{folder_to_delete}' does not exist.")

# Step 2 : Optional step : Pass 2A : Optional validation of N number of times.  

Set NUMBER_OF_VALIDATION_RUNS_REQUESTED
*   N = 0 to skip this validation  (Can yeild low accuacacy of Wiki)
*   N = 1 (default) . Validate once
*   N = 2 . Validate twice (Better accuracy of LLM)
*   N = 3.  Recommended for accuracy of LLM WIki Router. Solid validation.



In [ ]:
import shutil

# =====================================================================
# 🏙️ STEP 6.2 PROMPT STRINGS & SYSTEM INSTRUCTIONS (GLOBAL TEXT CONSTANTS)
# =====================================================================

SYSTEM_INSTRUCTION_2A = """You are a paranoid cross-document audit linter. Your single, critical mission is to ensure that no raw information is hidden from the upcoming wiki materialization loop.

Your failure condition is omission, NOT over-inclusion. If an attached raw file contains even a single sentence, paragraph,  metric, age limit, or guideline that touches upon a topic's scope, you MUST append that filename string into the topic's 'contributing_raw_files' array.

It is perfectly acceptable and safe to err on the side of including an extra file in the list.
It is completely fatal to the accuracy of the downstream WhatsApp chatbot if a raw file containing critical data is left out of a topic container. Audit the current configuration string and output a comprehensive, updated map."""

TEMPLATE_2A = """You are auditing the relational file-to-topic mapping configuration for a public enduser facing Question Answering chatbot wiki vault.

You are provided with two primary context layers:
1. The attached ground-truth raw markdown source documents.
2. The current unvalidated taxonomy blueprint JSON configuration string listed below.

=== CURRENT TAXONOMY MAP STRING ===
{current_json_map_string}
===================================

Your task is to re-evaluate every single master topic container listed in the JSON schema block. Actively read the attached raw files to discover hidden data connections.

If you locate ANY raw file that contains information relevant to a topic's scope but is currently missing from its 'contributing_raw_files' list, patch the array by appending that explicit file string immediately. Prioritize comprehensive coverage and over-inclusion over brevity.

Output your refined data mapping strictly within the requested JSON schema contract layout."""


# =====================================================================
# 🏙️ STEP 6.2 UTILITY TEMPLATE BLOCK
# =====================================================================

def get_pass_2a_prompt(current_blueprint_string: str) -> str:
    """Injects the current text layout of the taxonomy JSON string into the auditor template."""
    return TEMPLATE_2A.format(current_json_map_string=current_blueprint_string)


# =====================================================================
# 🏙️ STEP 6.2 EXECUTION BLOCK: THE PASS 2A MAPPING AUDITOR MATRIX
# =====================================================================

def execute_pass_2a_mapping_auditor(
    raw_dir: str,
    pass1_dir: str,
    pass2a_dir: str,
    validation_runs_requested: int
) -> dict:
    """
    Executes Pass 2A: The Mapping Auditor.
    Safely clones the Pass 1 data map, then run a sequential audit cascade loop (0, 1, or 3 times)
    to check for file omissions against the 14 raw source files.
    """
    print(f"\n🎬 Initializing [Pass 2A] Mapping Auditor Engine Layer...")
    os.makedirs(pass2a_dir, exist_ok=True)

    source_blueprint_path = os.path.join(pass1_dir, "master_taxonomy_blueprint.json")
    target_blueprint_path = os.path.join(pass2a_dir, "master_taxonomy_blueprint.json")

    # Structural Safeguard Check: Verify that the source blueprint file exists
    if not os.path.exists(source_blueprint_path):
        raise FileNotFoundError(f"❌ Pass 2A Fault: Missing master blueprint file path at '{source_blueprint_path}'")

    # Step 1: Base Step — Always clone the file to Pass2AOutput directory immediately
    print(f"📁 Base Step: Copying blueprint from {source_blueprint_path} directly to {target_blueprint_path}...")
    shutil.copy(source_blueprint_path, target_blueprint_path)

    # Load the baseline data structure into our working memory dictionary container
    working_blueprint_text = read_md(target_blueprint_path)
    current_validated_dict = json.loads(working_blueprint_text)

    # Step 2: Parameter-Driven Loop Logic Gate
    if validation_runs_requested <= 0:
        print("⏭️ [Pass 2A Status]: 'validation_runs_requested' set to 0. Bypassing API audit entirely. Utilizing cloned map.")
        print("📊 [Pass 2A Result]: SUCCESS. Cloned roadmap file ready for Pass 2 loop execution.")
        return current_validated_dict

    # Harvest ground-truth raw source files for the context verification turn
    raw_filenames = list_md(raw_dir)
    print(f"📦 Gathering context for verification: Packaging {len(raw_filenames)} raw files into text Part array chunks...")
    all_raw_file_parts = [wrap_file_with_boundary(raw_dir, f) for f in raw_filenames]

    # Execute the cascading audit loop runs sequentially (1 or 3 times)
    for run_index in range(1, validation_runs_requested + 1):
        print(f"\n🔄 Running Pass 2A Verification Audit Cycle [{run_index}/{validation_runs_requested}] via Gemini 2.5 Pro...")

        # Serialize current memory dictionary state into plain text for model ingestion
        current_map_string = json.dumps(current_validated_dict, indent=2)
        prompt_pass_2a = get_pass_2a_prompt(current_map_string)

        # Dispatch model call under our reused MasterTaxonomyBlueprint contract
        response_2a = generateNEWNEWNEW(
            client_instance=client,
            custompromptstring=prompt_pass_2a,
            myparts=all_raw_file_parts, # Ingests all 14 ground-truth files for verification checking
            mymodel="gemini-2.5-pro",
            mytemperature=0,
            my_max_output_tokens=24000,
            expected_response_schema=MasterTaxonomyBlueprint, # Reuses identical Pydantic contract
            my_system_instruction=SYSTEM_INSTRUCTION_2A,
            my_tools=None
        )

        # Safely pull our dictionary results via the parse utility
        current_validated_dict = parse_response_to_dict(response_2a, MasterTaxonomyBlueprint)
        print(f"📊 [Cycle {run_index} Status]: SUCCESS. Verified dictionary returned safely.")

        # Write intermediate state changes straight to disk to prevent data memory drop faults
        with open(target_blueprint_path, "w", encoding="utf-8") as f_out:
            json.dump(current_validated_dict, f_out, indent=4, ensure_ascii=False)

        # Apply mandatory rate-limiting cooling delays between loop runs
        if run_index < validation_runs_requested:
            print("⏳ Applying 5-second rate-limiting cooling delay before next cycle dispatch...")
            time.sleep(5)

    print(f"\n🎉 [Pass 2A Pipeline Completion]: Comprehensive audit cascade completed with absolute integrity!")
    print(f"   💾 Final audited and locked master routing dictionary stored safely at: {target_blueprint_path}")
    return current_validated_dict


In [ ]:
# =====================================================================
# 🧪 RUNTIME NOTEBOOK EXECUTOR CELL: EXECUTE PASS 2A (NUMBER_OF_VALIDATION_RUNS_REQUESTED , N = 1 RUN)
# =====================================================================

# Enforce our targeted configuration strategy parameter
NUMBER_OF_VALIDATION_RUNS_REQUESTED = 2


# Define standard target workspace path variables matching our system layout rules
RAW_INPUT_DIR = SOURCE_RAW_DIR
PASS1_OUTPUT_DIR = OUTPUT_PLAN_DIR
PASS2A_OUTPUT_DIR = "/content/Pass2AOutput/"



try:
    print("🚦 Triggering Pass 2A Ingestion Audit Loop...")

    # Execute our newly established, decoupled Pass 2A mapping auditor matrix method
    audited_taxonomy_roadmap = execute_pass_2a_mapping_auditor(
        raw_dir=RAW_INPUT_DIR,
        pass1_dir=PASS1_OUTPUT_DIR,
        pass2a_dir=PASS2A_OUTPUT_DIR,
        validation_runs_requested=NUMBER_OF_VALIDATION_RUNS_REQUESTED
    )

    # 🔍 Post-Execution Metadata Preview Check inside your cell window
    print("\n🎉 ===================== AUDIT LOOP EXECUTED SUCCESSFULLY =====================")
    print(f"Final checked taxonomy stored safely at: {os.path.join(PASS2A_OUTPUT_DIR, 'master_taxonomy_blueprint.json')}")
    print("--------------------------------------------------------------------------------")
    print(f"Total compiled topic nodes verified on disk: {len(audited_taxonomy_roadmap.get('permitted_topics', []))}")

    print("\n📋 Dynamic Mapping Overview Preview (Topic -> contributing_raw_files):")
    for topic in audited_taxonomy_roadmap.get("permitted_topics", []):
        print(f" 🔹 [{topic['topic_slug']}]: {topic['contributing_raw_files']}")
    print("================================================================================\n")

except FileNotFoundError as file_missing_error:
    print(f"\n❌ Pre-execution Block Blocked: {str(file_missing_error)}")
    print("   Please ensure that Pass 1 completed successfully and generated the json roadmap file in your Colab path first.\n")

except Exception as system_run_fault:
    print(f"\n❌ Terminal Ingestion Error: Audit loop transaction failed. Reason: {str(system_run_fault)}")
    print("   Please double check your Vertex AI Project ID configuration boundaries inside your Step 2 setup cell.\n")


## Pass2B: Basic validation against hallunication of file names in the blueprint.json

In [ ]:
# =====================================================================
# 📁 STEP 6.3: PASS 2B — PURE PYTHON LOCAL LINK VALIDATOR ENGINE
# =====================================================================

def execute_pass_2b_local_link_validator(raw_dir: str, pass2a_dir: str) -> bool:
    """
    Acts as a pure Python sanity check linter with zero LLM API calls.
    Cross-references every filename listed in the Pass 2A JSON configuration
    directly against the physical markdown assets resting on the hard disk.

    Returns True ONLY if all tracking files match perfectly.
    """
    print("\n[Pass 2B] 🕵️ Commencing Pure-Python Local Link Validation Sanity Check...")

    blueprint_path = os.path.join(pass2a_dir, "master_taxonomy_blueprint.json")
    if not os.path.exists(blueprint_path):
        print(f"❌ Pass 2B Fault: Audited taxonomy map file not found at path '{blueprint_path}'")
        return False

    # 1. Read and load the audited blueprint map file into memory
    blueprint_text = read_md(blueprint_path)
    blueprint_dict = json.loads(blueprint_text)

    # 2. Harvest the literal files currently sitting on the hard drive
    actual_disk_files = set(list_md(raw_dir))
    print(f"[Pass 2B] Verified ground-truth folder footprint contains {len(actual_disk_files)} markdown documents.")

    total_links_checked = 0
    missing_file_anomalies = []

    topics_list = blueprint_dict.get("permitted_topics", [])

    # 3. Core Matrix Validation Cross-Check Loop
    for topic in topics_list:
        slug = topic.get("topic_slug")
        mapped_files = topic.get("contributing_raw_files", [])

        for filename in mapped_files:
            clean_name = filename.strip()
            total_links_checked += 1

            # Match the mapped string against the real filenames on disk
            if clean_name not in actual_disk_files:
                missing_file_anomalies.append({
                    "topic_slug": slug,
                    "hallucinated_name": clean_name
                })

    # 4. Integrity Routing Logic Gate
    print(f"[Pass 2B] Completed structural audit across {total_links_checked} relational tracking strings.")

    if len(missing_file_anomalies) > 0:
        print("\n💥 =================== PASS 2B INTEGRITY ANOMALIES DETECTED ===================")
        print(f"❌ Structural Fault: Found {len(missing_file_anomalies)} hallucinated raw file assignments.")
        print("   The upcoming materialization loop is BLOCKED from running to prevent info blackouts.")
        print("--------------------------------------------------------------------------------")
        for anomaly in missing_file_anomalies:
            print(f"  📍 Topic Slug Container: [{anomaly['topic_slug']}]")
            print(f"     Critical Error Error: Model assigned filename '{anomaly['hallucinated_name']}' which does not exist on disk!")
        print("===============================================================================\n")
        return False

    print("✅ [Pass 2B Status]: 100% Structural Consistency Verified! Zero filename hallucinations found.")
    print("   The audited configuration maps perfectly to disk assets. Safe to proceed to compilation loops.")
    return True


In [ ]:
# =====================================================================
# 🧪 RUNTIME NOTEBOOK EXECUTOR CELL: EXECUTE PASS 2B LINK VALIDATOR
# =====================================================================

# Re-use your existing ground-truth system constant variables:
# - SOURCE_RAW_DIR represents your 14 raw source files directory path.
# - PASS2A_OUTPUT_DIR represents the directory holding your audited JSON map.

try:
    print("🚦 Initializing Pass 2B Integrity Verification Scan...")

    # Execute the pure Python local link linter checker
    is_taxonomy_contract_valid = execute_pass_2b_local_link_validator(
        raw_dir=SOURCE_RAW_DIR,
        pass2a_dir=PASS2A_OUTPUT_DIR
    )

    # 🔍 Verification Evaluation Gate
    print("\n🎉 ===================== LINTER AUDIT RUN COMPLETED =====================")
    if is_taxonomy_contract_valid:
        print("📊 [System Verdict]: APPROVED FOR DEPLOYMENT!")
        print("   All mapped raw filenames match the hard drive perfectly. Zero string hallucinations.")
        print("   You are completely clear to launch the Pass 2 Materialization loop safely.")
    else:
        print("📊 [System Verdict]: CRITICAL DISK MISMATCH ERROR ENCOUNTERED!")
        print("   Pass 2B has blocked your compilation stream due to data inaccuracies.")
        print("   Please review the printed anomaly log above and fix the filename string typo.")
    print("========================================================================\n")

except Exception as check_error:
    print(f"\n❌ Pre-execution Verification Block Triggered an Exception: {str(check_error)}")
    print("   Please ensure that Pass 2A completed successfully and written the data roadmap file first.\n")


# Pass 2 (once both the above input validation is successful)

In [ ]:
# =====================================================================
# PART 1: SYSTEM INSTRUCTIONS & DYNAMIC TEMPLATE UTILITY FORMATTER
# =====================================================================
import json

SYSTEM_INSTRUCTION_2 = """You are a high-density knowledge extraction engine. Your objective is the absolute, verbatim replication of facts combined with airtight contextual guardrails.

CRITICAL SYSTEM COMPLIANCE MATRIX:
- ABSOLUTE SUCCESS CONDITION: You achieve 100% extraction recall of all metrics, parameters, and guidelines within the requested scope while prefixing every single bullet point, summary statement, table description row, FAQ entry, and QA pair with an explicit, unified bracketed metadata tag block format exactly matching: '[Target:: token1 | Milestone:: token2 | Action:: token3]'. The tag block must be the absolute first characters at the start of the line.
- ABSOLUTE FAILURE CONDITION: You omit a fact, generalize a number, contaminate a timeline, drift a metric value, place list markers (like '-' or '*') before the bracketed tag block, or vary the tags between a question and its corresponding answer. Writing an un-tagged statement or adding decorative icons means a complete system safety failure.

Write long, exhaustive, fact-contained markdown files with zero loose narrative prose or filler language. Stick strictly to the boundaries of the attached text files."""


In [ ]:
# =====================================================================
# PART 2: THE COMPREHENSIVE COMPLIANCE MASTER TEMPLATE (TEMPLATE_2)
# =====================================================================

TEMPLATE_2 = """Compile a 100% comprehensive, long-form Markdown wiki file for the following specialized Question Answering LLM Wiki chatbot topic container:

TARGET TOPIC DISPLAY HEADER: {display_name}
TARGET TOPIC FILE SLUG: {slug}.md
EXPLICIT SCOPE LIMITS: {scope}
EXCLUSION BOUNDARIES: {exclusions}

You must read the attached raw files and extract every single concrete fact, constraint,  guideline,  metric, and domain specific trigger rule.

You must map out the final file layout strictly using this exact template layout with zero decorative icons, emojis, or markdown blockquote symbols.
For your guidance, review the following exemplary one-shot blueprint of a correctly generated file optimized for downstream GPT lookups:

=== MANDATED ONE-SHOT EXEMPLAR FOR STRUCTURAL MIMICRY ===
---
topic_slug:  Topic
display_name: Display Name
contributing_sources: ["Filename.md"]
---

# Topic: TRIMESTER 1 CARE (WEEKS 1-12)

[CONTEXTUAL GUARDRAIL LOCK]:

## 1. Core Semantic Keywords
<keywords_block>
</keywords_block>

## 2. Executive Guidance & Context Layer
<executive_brief_block>
### A. Long Comprehensive Summary

### B. Explicit Architectural Scope Parameters

### C. Contrastive Boundary Analysis (What This Document Is NOT)

### D. Important Facts Summary

### E. Document Context & Validity Rules (When This Information Holds Good)

### F. Top 5 to 10 Most Popular FAQs
<qa_pair>
</qa_pair>
</executive_brief_block>

## 3. Exhaustive List of Facts
<facts_block>
</facts_block>

## 4. Granular Frequently Asked Questions (FAQs)
<secondary_faqs_block>
</secondary_faqs_block>

## 5. Target Question-and-Answer Matrix
<chatbot_lookup_matrix_block>
<qa_pair>
</qa_pair>
</chatbot_lookup_matrix_block>
=== END OF ONE-SHOT EXEMPLAR ===

Execute your extraction task now for the target topic using the attached raw source documents. Ensure you match the structure of the one-shot blueprint perfectly, maintaining absolute literal and mechanical fidelity.

CRITICAL MAPPING AND BRACKET PAIRING FORCE:
1. Every single individual row line generated across all sections MUST open with a flawlessly formed, completely closed bracket block: '[Target:: key1 | Milestone:: key2 | Action:: key3]'. You must NOT place hyphens (-), bullet asterisks (*), or numbered symbols before the bracketed block.
2. Every FAQ entry and QA pair MUST be wrapped inside separate, consecutive <qa_pair> and </qa_pair> tags.
3. For every line inside a <qa_pair> block, the bracketed key-value metadata text strings must be 100% identical and unchanged across both the question and the answer lines.

Output your response as clean markdown text within the lower_snake_case XML containers: <keywords_block>, <executive_brief_block>, <facts_block>, <secondary_faqs_block>, and <chatbot_lookup_matrix_block>. Complete all sections comprehensively without cutting off mid-sentence."""
print("✅ Part 2 initialized. TEMPLATE_2 is fully populated in memory.")


In [ ]:
def get_pass_2_prompt(topic_blueprint: dict) -> str:
    """
    Extracts variable metadata components from an individual topic's audited dictionary contract
    and formats them cleanly into the global structural triple-quoted template instruction string.

    Optimized for downstream GPT lookups by stripping out confusing '#' prefix structures,
    cleaning string cases, and generating clean, comma-separated metadata token strings.
    """
    # Programmatically serialize the seed keywords array into flat, lowercase alphanumeric tokens
    # Separated cleanly by commas for unambiguous parsing by downstream GPT engines
    hashtag_tags_list = topic_blueprint.get("target_seed_keywords", [])
    keyword_string = ", ".join([str(tag).strip().lower().replace("#", "") for tag in hashtag_tags_list])

    # Extract structural text strings cleanly, using non-empty safe fallback defaults if fields are missing
    display_name = topic_blueprint.get("display_name", "Unnamed Topic")
    slug = topic_blueprint.get("topic_slug", "unnamed_topic")
    scope = topic_blueprint.get("scope_definition", "No explicit scope recorded.")
    exclusions = topic_blueprint.get("exclusion_rules", "No explicit exclusion rules recorded.")
    mapped_sources_list = topic_blueprint.get("contributing_raw_files", [])

    # Return the clean, interpolated prompt layout matching our exact template keys
    return TEMPLATE_2.format(
        display_name=display_name,
        slug=slug,
        scope=scope,
        exclusions=exclusions,
        sources_json=json.dumps(mapped_sources_list),
        keyword_tags_string=keyword_string
    )


In [ ]:
# =====================================================================
# 🏗️ STEP 7 EXECUTION MODULE: THE INDIVIDUAL TOPIC COMPILER LOOP
# =====================================================================

def execute_pass_2_materialization_loop(raw_dir: str, pass2a_dir: str, wiki_dir: str):
    """
    Loops programmatically through the audited taxonomy roadmap matrix from Pass 2A.
    Ingests ONLY the specific mapped raw files for each topic block and writes
    the finalized, comprehensive wiki pages straight to the local hard disk directory.
    """
    # 1. Resolve and check the audited ground-truth json file path generated by Pass 2A
    blueprint_file_path = os.path.join(pass2a_dir, "master_taxonomy_blueprint.json")
    if not os.path.exists(blueprint_file_path):
        raise FileNotFoundError(
            f"❌ Pass 2 Ingestion Fault: Missing audited master blueprint file at path '{blueprint_file_path}'. "
            "Please ensure Pass 2A ran successfully before triggering."
        )

    # Read raw text data from disk and parse into a dictionary container natively
    blueprint_text = read_md(blueprint_file_path)
    taxonomy_blueprint_dict = json.loads(blueprint_text)

    topics_list = taxonomy_blueprint_dict.get("permitted_topics", [])
    if not topics_list:
        print("⚠️ Pass 2 Ingestion Alert: No permitted topic items located inside the taxonomy matrix dictionary.")
        return

    print(f"     [Pass 2 Loop] Audited blueprint loaded. Preparing compilation runs for {len(topics_list)} topic containers...")
    os.makedirs(wiki_dir, exist_ok=True)

    # 2. Process each independent topic container sequentially
    for current_topic in topics_list:
        slug = current_topic["topic_slug"]
        display_name = current_topic["display_name"]
        mapped_files = current_topic["contributing_raw_files"]

        print(f"     📄 Processing Topic File Page: {slug}.md ...")
        print(f"        Loading context via targeted gatekeeper loader array: {mapped_files}")

        # ⚡ TARGETED INGESTION GATEKEEPER: Open ONLY the requested raw text file strings into memory parts
        topic_specific_parts = package_targeted_sources(raw_dir, mapped_files)

        if not topic_specific_parts:
            print(f"        ⚠️ Ingestion notice: Generation skipped for '{slug}.md' due to zero source file matches on disk.")
            continue

        # Format custom user prompt templates out of the global text constant containers
        prompt_pass_2 = get_pass_2_prompt(current_topic)

        # 3. Dispatch the compilation pass turn via Gemini 2.5 Flash for rapid text processing loops
        response_p2 = generateNEWNEWNEW(
            client_instance=client,
            custompromptstring=prompt_pass_2,
            myparts=topic_specific_parts,        # Injects ONLY the required text file parts
            mymodel="gemini-2.5-pro",
            mytemperature=0,
            my_max_output_tokens=48000,
            expected_response_schema=None,       # Returns plain text Markdown strings directly
            my_system_instruction=SYSTEM_INSTRUCTION_2,
            my_tools=None
        )

        # 4. Programmatically save the unified markdown topic asset directly to the wiki vault folder
        target_file_output_path = os.path.join(wiki_dir, f"{slug}.md")
        write_md(target_file_output_path, response_p2.text)
        print(f"        ✅ Materialization successful: {target_file_output_path}")


        print('Sleeping for 10 sec.....Rate limiting')
        # Brief cooling delay to keep loop transactions steady inside Google Colab cells
        time.sleep(60)

    print("     [Pass 2 Loop] All independent markdown topic pages successfully written to the vault directory.")


In [ ]:
# =====================================================================
# 🏗️ STEP 7 RUNTIME BLOCK: PASS 2 MATERIALIZATION LOOP COORDINATOR
# =====================================================================

def run_pass_2_materialization_pipeline(
    raw_dir: str,
    pass2a_dir: str,
    wiki_dir: str
) -> bool:
    """
    Coordinates the Phase 2 compilation sequence.
    First runs Pass 2B as an absolute validation gate. If files match perfectly,
    it executes the Pass 2 materialization loop to compile unified topic assets.
    """
    print("\n🎬 Initializing [Pass 2 Lifecycle] Compilation Module...")

    # 1. Trigger Pass 2B: Local Link Validator to check filename assignments
    is_taxonomy_contract_valid = execute_pass_2b_local_link_validator(
        raw_dir=raw_dir,
        pass2a_dir=pass2a_dir
    )

    # 2. Strict Conditional Guard Gate Checklist
    if not is_taxonomy_contract_valid:
        print("\n💥 =================== PASS 2 LIFECYCLE EXECUTION HALTED ===================")
        print("❌ Critical System Alert: Pass 2B has flagged file mapping anomalies on disk.")
        print("   The materialization loop is BLOCKED from running to prevent data gaps.")
        print("   Please fix the filename mismatch typos listed in the log above.")
        print("===========================================================================\n")
        return False

    print("🚀 [Validation Passed]: All folder taxonomy records match physical disk files.")
    print("   Launching the individual topic compilation loop now...")

    try:
        # 3. Transfer control smoothly down to the verified materialization loop handler
        execute_pass_2_materialization_loop(
            raw_dir=raw_dir,
            pass2a_dir=pass2a_dir,
            wiki_dir=wiki_dir
        )
        print("\n🎉 [Pass 2 Lifecycle Status]: SUCCESS. All topic assets materialized on harddrive.")
        return True

    except Exception as loop_fault:
        print(f"\n❌ Pass 2 Loop Execution Fault: Processing transaction failed. Reason: {str(loop_fault)}")
        return False


In [ ]:
pausehere

In [ ]:
COMPILED_WIKI_DIR = "/content/LLM_Wiki_Vault/wiki/"


In [ ]:
# =====================================================================
# 🧪 RUNTIME NOTEBOOK EXECUTOR CELL: TRANSACTING PASS 2 COMPILATION
# =====================================================================

# Re-use existing system constant variables from your active runtime space:
# - SOURCE_RAW_DIR represents your 14 raw source files folder path.
# - PASS2A_OUTPUT_DIR represents the folder holding your audited JSON map.

# Establish the clear, unified target wiki vault deployment folder path

try:
    # Trigger the integrated, gate-protected Pass 2 compilation pipeline
    is_compilation_successful = run_pass_2_materialization_pipeline(
        raw_dir=SOURCE_RAW_DIR,          # Global constant input path
        pass2a_dir=PASS2A_OUTPUT_DIR,    # Global constant input path
        wiki_dir=COMPILED_WIKI_DIR       # Target output vault landing zone
    )

    if is_compilation_successful:
        print("\n📬 Ingestion loop finalized. Your wiki files are fully written.")
        print(f"   Check your folder sidebar layout path: {COMPILED_WIKI_DIR}")

except Exception as global_pipeline_fault:
    print(f"\n❌ Pipeline Halted by an unexpected system exception: {str(global_pipeline_fault)}")


In [ ]:
pause here

#Step 3: Index creation after all topic files are creation

In [ ]:
# =====================================================================
# PART 1: PASS 3A & PASS 3B PROMPT STRINGS (GLOBAL TEXT CONSTANTS)
# =====================================================================

SYSTEM_INSTRUCTION_3A = """You are a strategic taxonomy index organizer. Your sole purpose is to compile a highly compact, low-token master folder map. Write short, dense markdown entries with zero decorative prose, icons, or emojis."""

PROMPT_3A = """Analyze the enclosed finalized wiki topic documents sitting in the local directory.

Your task is to generate a concise, high-level master directory map file layout with zero decorative icons, emojis, or markdown blockquotes. Group the topic files under broad operational categories. For each file reference, provide its exact filename path, its display header name, and a space-separated list of core target tokens.

Output your response strictly matching the formatting structure of this design pattern template layout:
# Master Category Index

* Path: topic_1.md | Title: Topic 1  | Target:
* Path: topic_2.md | Title: Topic 2 | Target:

Do not reference or cite old raw document assets here. Output strictly as clean, standard markdown text."""



In [ ]:
SYSTEM_INSTRUCTION_3B = """You are a precision technical document indexer mapping relational metadata loops for a high-stakes query routing engine. Your single objective is to translate flat text files into strict, un-ambiguous key-value attribute manifest tables."""

TEMPLATE_3B = """You are building the definitive lookup map for an enterprise public health chatbot query router.

Review the attached high-level master category directory index (index.md) provided below:
=== MASTER index.md TEXT LAYOUT ===
{index_md_content}
===================================

You must also evaluate the full underlying compiled topic documents attached as multi-part elements. Your task is to output a deeply granular, comprehensive semantic index file layout. For every single topic document path discovered in the directory, you must construct a strict, standardized text block featuring a high-density Attribute Manifest Table.

You must map out each file's boundaries using zero decorative icons, emojis, or markdown blockquotes, matching this exact structural anatomy:

### File Path Target: trimester_1_care.md
<routing_manifest>

| System Attribute Parameter | Explicit Manifest Logic Constraints |
| :--- | :--- |
| Valid Target Profile | Target:: pregnant_mother |
| Active Milestone Window | Milestone:: trimester1, weeks_1_12 |
| Allowed Structural Keywords | Keywords:: trimester_1, first_trimester, anc_1, folic_acid, nausea, early_pregnancy, initial_screenings |
| Core Symptom Vectors | Conditions:: nausea, vomiting, early_fatigue, constipation |
| Absolute Invalidation Bounds | Exclusions:: trimester2, trimester3, infant_care, deworming_protocols, ifa_combination_tablets |
| Internal Structural Anchors | Headers:: ## 1. Core Semantic Keywords, ## 2. Executive Guidance & Context Layer, ## 3. Exhaustive List of Facts |
</routing_manifest>

CRITICAL ROUTING ACCURACY MANDATE:
1. The 'Absolute Invalidation Bounds' table row MUST explicitly list all terms, stage timelines, and clinical scenarios that are strictly forbidden from entering that specific file container based on its exclusion rules.
2. The 'Internal Structural Anchors' row must list the exact, literal heading text found inside that topic file.
3. All parameters must use flat, lowercase alphanumeric tokens separated cleanly by commas.

Output your response as clean markdown text. Do not include loose narrative chat outside the template boundaries."""

print("✅ Part 1 completed. Pass 3A and Pass 3B text prompt constants registered successfully.")


In [ ]:
# =====================================================================
# PART 2: PASS 3A & PASS 3B COMPILATION FUNCTIONS LABELS
# =====================================================================

def execute_pass_3a_category_indexing(wiki_dir: str) -> str:
    """
    Pass 3A: Scans the /wiki/ directory, reads the newly generated topic pages,
    and compiles a compact category overview layout written to wiki/index.md.
    """
    print("\n[Pass 3A] 🔄 Commencing Tier 1 Concise Index Generation Pass...")

    # 1. Harvest physical wiki files from disk, excluding previous index layers
    wiki_filenames = list_md(wiki_dir)
    active_topic_files = [f for f in wiki_filenames if f not in ["index.md", "index_detailed.md", "log.md"]]

    if not active_topic_files:
        raise FileNotFoundError(f"❌ Pass 3A Fault: No materialized topic markdown files located inside '{wiki_dir}'.")

    print(f"[Pass 3A] Packaging {len(active_topic_files)} verified topic files into text Part array chunks...")
    compiled_wiki_parts = [wrap_file_with_boundary(wiki_dir, wf) for wf in active_topic_files]

    # 2. Dispatch model turn via Gemini 2.5 Pro for global taxonomy layout analysis
    response_3a = generateNEWNEWNEW(
        client_instance=client,
        custompromptstring=PROMPT_3A,
        myparts=compiled_wiki_parts,
        mymodel="gemini-2.5-pro",
        mytemperature=0,
        my_max_output_tokens=45000,
        expected_response_schema=None,
        my_system_instruction=SYSTEM_INSTRUCTION_3A,
        my_tools=None
    )

    # 3. Commit text straight to disk asset folder path
    target_index_path = os.path.join(wiki_dir, "index.md")
    write_md(target_index_path, response_3a.text)
    print(f"[Pass 3A] ✅ Tier 1 concise category index successfully written to harddrive: {target_index_path}")
    return response_3a.text


def execute_pass_3b_semantic_indexing(wiki_dir: str, index_md_text: str) -> str:
    """
    Pass 3B: Ingests index.md and topic contents to build a granular,
    manifest-driven lookup matrix table written to wiki/index_detailed.md.
    """
    print("[Pass 3B] 🔄 Commencing Tier 2 Detailed Semantic Index Matrix Compilation Pass...")

    # 1. Harvest active wiki files from disk again
    wiki_filenames = list_md(wiki_dir)
    active_topic_files = [f for f in wiki_filenames if f not in ["index.md", "index_detailed.md", "log.md"]]
    compiled_wiki_parts = [wrap_file_with_boundary(wiki_dir, wf) for wf in active_topic_files]

    # 2. Wrap the newly generated index.md string into an isolated text Part boundary element
    index_md_part = types.Part.from_text(text=f"--- GROUND TRUTH CATEGORY MASTER INDEX (index.md) ---\n{index_md_text}\n")
    pass3b_inputs = compiled_wiki_parts + [index_md_part]

    # Template index_md_text directly into the prompt payload constant container
    prompt_pass_3b = TEMPLATE_3B.format(index_md_content=index_md_text)

    # 3. Dispatch model turn via Gemini 2.5 Pro to manage high-density matrix attribute binding
    response_3b = generateNEWNEWNEW(
        client_instance=client,
        custompromptstring=prompt_pass_3b,
        myparts=pass3b_inputs,
        mymodel="gemini-2.5-pro",
        mytemperature=0,
        my_max_output_tokens=48000,
        expected_response_schema=None,
        my_system_instruction=SYSTEM_INSTRUCTION_3B,
        my_tools=None
    )

    # 4. Commit master routing matrix sheet straight to disk
    target_detailed_path = os.path.join(wiki_dir, "index_detailed.md")
    write_md(target_detailed_path, response_3b.text)
    print(f"[Pass 3B] ✅ Tier 2 detailed semantic lookup index successfully written to harddrive: {target_detailed_path}")
    return response_3b.text

print("✅ Part 2 initialized. execute_pass_3a and execute_pass_3b execution methods compiled safely.")


In [ ]:
# =====================================================================
# PART 3: REUSABLE MASTER INDEX LIFECYCLE COORDINATOR
# =====================================================================

# Re-use existing system constant variables from your active notebook session
# COMPILED_WIKI_DIR represents your generated markdown wiki folder path (/content/LLM_Wiki_Vault/wiki/)
try:
    print("🚦 Triggering Phase 3 Dual Index Generation Module...")

        # Apply a safe pacing cooling delay between transactions
    print("⏳ Applying 30-second rate-limiting cooling delay...")
    time.sleep(60)

    # 1. Execute Pass 3A to build index.md
    index_concise_content = execute_pass_3a_category_indexing(wiki_dir=COMPILED_WIKI_DIR)

    # Apply a safe pacing cooling delay between transactions
    print("⏳ Applying 30-second rate-limiting cooling delay...")
    time.sleep(30)

    # 2. Execute Pass 3B to build index_detailed.md featuring the manifest tables
    index_detailed_content = execute_pass_3b_semantic_indexing(
        wiki_dir=COMPILED_WIKI_DIR,
        index_md_text=index_concise_content
    )

    print("\n🎉 [Phase 3 Indexing Lifecycle Complete]: System lookup matrices successfully settled!")
    print(f"   📁 Low-Token Lane Router written directly to: {os.path.join(COMPILED_WIKI_DIR, 'index.md')}")
    print(f"   📋 Manifest Relational Table Map written directly to: {os.path.join(COMPILED_WIKI_DIR, 'index_detailed.md')}")
    print("   Your Knowledge Vault is fully searchable and optimized for downstream GPT routing calls.")

except Exception as indexing_fault:
    print(f"\n❌ Phase 3 Indexing Pipeline Halted by an error exception: {str(indexing_fault)}")
    print("   Please ensure Phase 2 materialization ran successfully and generated topic pages before executing.")


# Pass 4: Check filename halluncination

In [ ]:
import os
import re
from typing import List, Set

# =====================================================================
# 📁 STEP 10: PASS 4 — PURE-PYTHON INDEX FILENAME AUDITOR ENGINE
# =====================================================================

def execute_pass_4_index_filename_auditor(wiki_dir: str) -> bool:
    """
    Acts as a pure-Python compiler safety gate with zero LLM API costs.
    Extracts all markdown file paths referenced inside index.md and index_detailed.md,
    and cross-references them directly against physical hard disk file states.

    Returns True ONLY if all index paths are verified as real, on-disk assets.
    """
    print("\n[Pass 4] 🕵️ Commencing Pure-Python Index Filename Graph Audit...")

    index_path = os.path.join(wiki_dir, "index_stage_routing_protocol.md")
    detailed_path = os.path.join(wiki_dir, "index_detailed.md")

    # 1. Structural Pre-flight Checks: Verify index files exist before auditing
    if not os.path.exists(index_path):
        print(f"❌ Pass 4 Fault: Layer 1 Concise Index sheet not found at path '{index_path}'")
        return False


    # 2. Harvest the literal topic files currently sitting on the hard drive
    # Filters out index files themselves to build a clean ground-truth set
    all_disk_files = set(list_md(wiki_dir))
    core_topic_disk_files = {f for f in all_disk_files if f not in [ "index_stage_routing_protocol.md", "log.md"]}

    print(f"[Pass 4] Ground-truth folder footprint contains {len(core_topic_disk_files)} compiled topic markdown pages.")

    # 3. Read index sheet text blocks into working memory containers
    index_text = read_md(index_path)

    # 4. Regex Parsing Machine: Extract any alphanumeric string block ending in .md
    # Enforces explicit pattern extraction matching for markdown links and manifest blocks
    path_regex_pattern = r"([a-zA-Z0-9_]+\.md)"

    referenced_in_index: Set[str] = set(re.findall(path_regex_pattern, index_text))
    referenced_in_detailed: Set[str] = set(re.findall(path_regex_pattern, detailed_text))

    # Clean out self-references to focus purely on topic file allocations
    for self_file in ["index_stage_routing_protocol.md",   "log.md"]:
        referenced_in_index.discard(self_file)

    print(f"[Pass 4] Extracted {len(referenced_in_index)} distinct topic file links from index.md")

    graph_anomalies_logged = []

    # 5. Core Cross-Check Loop: Verify index references exist physically on disk
    for ref_file in referenced_in_index:
        if ref_file not in core_topic_disk_files:
            graph_anomalies_logged.append({
                "source_index": "index_stage_routing_protocol.md",
                "hallucinated_link": ref_file
            })

    for ref_file in referenced_in_detailed:
        if ref_file not in core_topic_disk_files:
            graph_anomalies_logged.append({
                "source_index": "index_stage_routing_protocol.md",
                "hallucinated_link": ref_file
            })

    # 6. Orphan Detection Check: Warn if a compiled file is completely missing from the indexes
    all_index_references = referenced_in_index.union(referenced_in_detailed)
    orphan_files_found = core_topic_disk_files.difference(all_index_references)

    # 7. Integrity Routing Logic Gate Decisions
    if len(graph_anomalies_logged) > 0 or len(orphan_files_found) > 0:
        print("\n💥 =================== PASS 4 ARCHITECTURAL GRAPH ANOMALIES ===================")
        print(f"❌ Structural Fault: Detected link irregularities inside the indexing layers.")
        print("   The generated Knowledge Vault is UNSTABLE. Production deployment is blocked.")
        print("--------------------------------------------------------------------------------")

        if graph_anomalies_logged:
            print("🚨 CRITICAL ERRORS: Hallucinated/Broken File Path References Found:")
            for error in graph_anomalies_logged:
                print(f"  📍 Inside Index File: [{error['source_index']}] -> References filename '{error['hallucinated_link']}' which does not exist on disk!")

        if orphan_files_found:
            print("\n⚠️ SYSTEM WARNING: Orphaned Topic Files Detected (Compiled on disk but omitted from indexes):")
            for orphan in orphan_files_found:
                print(f"  📍 Missing Node: '{orphan}' is physically written but has zero mapping links inside index files.")
        print("================================================================================\n")
        return False

    print("✅ [Pass 4 Status]: 100% Index Path Integrity Verified! Zero filename hallucinations found.")
    print("   All index mapping lines correspond perfectly to verified physical files on the hard drive.")
    print("   Your compiled offline LLM Wiki Vault is officially sealed and certified safe for runtime deployment.")
    return True


In [ ]:
# =====================================================================
# 🧪 RUNTIME NOTEBOOK EXECUTOR CELL: TRANSACTING PASS 4 AUDIT SCAN
# =====================================================================

# Re-use existing system constant variables from your active runtime session
# COMPILED_WIKI_DIR maps directly to your generated vault directory (/content/LLM_Wiki_Vault/wiki/)

try:
    print("🚦 Initializing Pass 4 Pipeline Graph Validation Check...")

    # Trigger the pure Python index file layout path checker linter function
    is_vault_certified_safe = execute_pass_4_index_filename_auditor(wiki_dir=COMPILED_WIKI_DIR)

    print("\n🎉 ===================== COGNITIVE VAULT PIPELINE RUN COMPLETE =====================")
    if is_vault_certified_safe:
        print("📊 [System Compilation Verdict]: APPROVED FOR EXPORT!")
        print("   Your local markdown wiki files and manifest indices match perfectly with zero typos.")
        print("   You are 100% clear to copy your /wiki/ files over to your new runtime QA system notebook safely.")
    else:
        print("📊 [System Compilation Verdict]: REJECTED BY AUTOMATED AUDITOR GATE!")
        print("   The local index files contain reference path typos that will cause runtime app crashes.")
        print("   Please trace the printed anomalies logged above to align your taxonomy filenames.")
    print("====================================================================================\n")

except Exception as audit_run_fault:
    print(f"\n❌ Audit Pipeline Blocked by an unhandled system exception: {str(audit_run_fault)}")
    print("   Please ensure that Phase 3 indexers completed running and populated files on disk first.\n")


In [ ]:
ffssf

In [ ]:
import shutil
import os

source_directory = '/content/LLM_Wiki_Vault/wiki'
output_base_name = 'Export/Wiki'

# Ensure the output directory exists
output_dir = os.path.dirname(output_base_name)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

try:
    shutil.make_archive(output_base_name, 'zip', source_directory)
    print(f"Successfully created zip archive: {output_base_name}.zip")
except Exception as e:
    print(f"Error creating zip archive: {e}")

 # Extra TOC

In [ ]:
# =====================================================================
# 🏗️ CODESET CELL 1: PASS 3C — BILINGUAL INTENT PROMPTS (GLOBAL CONSTANTS)
# =====================================================================

SYSTEM_INSTRUCTION_3C = """You are an elite multi-lingual taxonomy data architect. Your sole objective is to compile a localized multi-lingual intent mapping matrix file. Your response must contain zero decorative prose, blockquotes, or conversational chatter outside of standard markdown headers and punchy bullet entries."""

PROMPT_3C = """Analyze the enclosed finalized wiki topic documents sitting in the local directory alongside the master index.md category layout text.

Your critical task is to generate a third, dedicated Table ofContents asset file named 'index_bilingual_intent.md' to serve as a high-precision linguistic translation bridge for a downstream runtime chatbot. You must map conversational Hindi and Hinglish phrases/root words directly to their valid target file path arrays.

CRITICAL 'SAFE OVER-POOLING' MANDATE:
1. SAFE OVER-INCLUSION FOR GENERAL INTENTS: If an intention keyword or conversational phrase is broad, open-ended, or lacks an explicit gestational week or childhood age parameter (e.g., 'Pregnancy me kya khana chahiye', 'pregancny me konsi sabji khau', 'bacha kamzor h'), you MUST err on the side of comprehensiveness. You are strictly REQUIRED to group and suggest a complete pool of all relevant baseline chronological files simultaneously (e.g., matching 'khana' to Trimester 1, 2, and 3 files together) so that no nutritional guideline is omitted.
2. CONVERSATIONAL VECTORS FILTERING: Actively look through the text contents of the topic pages to isolate common Hinglish terms used in field logs (such as 'tika', 'jaanch', 'paisa', 'safed pani', 'dard', 'wajan') and map them to their corresponding clinical or administrative file containers.

Output your response strictly matching the formatting structure of this design pattern template layout, using zero emojis or loose text outside the blocks:

# Master Bilingual Intent Intersection Matrix

### 1. Maternal Nutrition & Diet (गर्भावस्था आहार, पोषण और जीवनशैली)
* Intent Phrases: kya khana chahiye, konsi sabji khaye, fal khana hai, kya khaye, diet chart, kitna aaram kare, bhojan
* Target File Pool Array: ['maternal_care_pre_conception.md', 'maternal_care_trimester_1.md', 'maternal_care_trimester_2.md', 'maternal_care_trimester_3_and_birth_prep.md']

### 2. Acute Clinical Symptoms & Severities (गर्भावस्था के लक्षण और आपातकालीन स्थिति)
* Intent Phrases: white discharge, safed pani aata hai, pair me soojan, chakkar aa raha, pet me dard, khoon kam hai, 7 point khoon
* Target File Pool Array: ['maternal_anemia_diagnosis_and_treatment.md', 'maternal_danger_signs_and_emergencies.md']

Do not reference old raw source files. Output strictly as clean, standard markdown plain text using single quotes inside the target arrays."""

print("✅ Cell 1 completed. Pass 3C text prompt constants registered successfully for 22 topics.")


In [ ]:
 # =====================================================================
# 🏗️ CODESET CELL 2: PASS 3C — EXECUTION METHOD & STORAGE WRAPPER
# =====================================================================

def execute_pass_3c_bilingual_intent_indexing(wiki_dir: str, index_md_text: str) -> str:
    """
    Pass 3C: Ingests index.md and topic contents to materialize the dynamic
    semantic translation bridge index 'index_bilingual_intent.md' on disk.
    """
    print("\n[Pass 3C] 🔄 Commencing Tier 3 Bilingual Intent Lookup Index Generation Pass...")

    # 1. Harvest active wiki topic markdown files from disk
    wiki_filenames = list_md(wiki_dir)
    active_topic_files = [f for f in wiki_filenames if f not in ["index.md", "index_detailed.md", "index_bilingual_intent.md", "log.md"]]
    compiled_wiki_parts = [wrap_file_with_boundary(wiki_dir, wf) for wf in active_topic_files]

    # 2. Package index.md plain text into an isolated text Part boundary element
    index_md_part = types.Part.from_text(text=f"--- GROUND TRUTH MASTER INDEX (index.md) ---\n{index_md_text}\n")
    pass3c_inputs = compiled_wiki_parts + [index_md_part]

    # 3. Dispatch the transaction turn through Gemini 2.5 Pro to build your bilingual pooling structures
    response_3c = generateNEWNEWNEW(
        client_instance=client,
        custompromptstring=PROMPT_3C,
        myparts=pass3c_inputs,
        mymodel="gemini-2.5-pro",
        mytemperature=0,
        my_max_output_tokens=40000,
        expected_response_schema=None,
        my_system_instruction=SYSTEM_INSTRUCTION_3C,
        my_tools=None
    )

    # 4. Commit the new billingual translation map straight to disk
    target_intent_path = os.path.join(wiki_dir, "index_bilingual_intent.md")
    write_md(target_intent_path, response_3c.text)
    print(f"[Pass 3C] ✅ Tier 3 bilingual intent index successfully written to harddrive: {target_intent_path}")
    return response_3c.text

print("✅ Cell 2 initialized. execute_pass_3c_bilingual_intent_indexing compiled safely.")


In [ ]:
# =====================================================================
# 🏗️ CODESET CELL 3: EXTENDED LIFECYCLE COORDINATOR (3-INDEX EDITION)
# =====================================================================

try:
    print("🚦 Triggering Phase 3 Extended Triple Index Generation Module...")

    # 1. Execute Pass 3A to build index.md
    index_concise_content = execute_pass_3a_category_indexing(wiki_dir=COMPILED_WIKI_DIR)
    print("⏳ Applying 15-second rate-limiting cooling delay...")
    time.sleep(60)

    # 2. Execute Pass 3B to build index_detailed.md featuring the manifest tables
    index_detailed_content = execute_pass_3b_semantic_indexing(
        wiki_dir=COMPILED_WIKI_DIR,
        index_md_text=index_concise_content
    )
    print("⏳ Applying 15-second rate-limiting cooling delay...")
    time.sleep(60)

    # 3. Execute Pass 3C to materialize index_bilingual_intent.md on disk
    index_intent_content = execute_pass_3c_bilingual_intent_indexing(
        wiki_dir=COMPILED_WIKI_DIR,
        index_md_text=index_concise_content
    )

    print("\n🎉 [Phase 3 Indexing Lifecycle Complete]: All 3 lookup matrices successfully settled on disk!")
    print(f"   📁 Tier 1 (Lane Selector Map): {os.path.join(COMPILED_WIKI_DIR, 'index.md')}")
    print(f"   📋 Tier 2 (Clinical Manifest Tables): {os.path.join(COMPILED_WIKI_DIR, 'index_detailed.md')}")
    print(f"   📋 Tier 3 (Bilingual Intent Map with Over-Pooling): {os.path.join(COMPILED_WIKI_DIR, 'index_bilingual_intent.md')}")
    print("   Your Knowledge Vault is fully multi-lingual searchable and optimized for downstream GPT routing calls.")

except Exception as indexing_fault:
    print(f"\n❌ Phase 3 Indexing Pipeline Halted by an error exception: {str(indexing_fault)}")
    print("   Please ensure Phase 2 materialization ran successfully before executing.")


In [ ]:
# =====================================================================
# 🏗️ PASS 3A PROMPT PATCH: IMMUTABLE 3-SECTION PROTOCOL COMPILER
# NAND WORKING TESTED
# =====================================================================

SYSTEM_INSTRUCTION_3A = """You are an strategic taxonomy index organizer.
You are an expert in strategic thinking. You can also good in modelling and analyzing how LLMs can do multi-step reasoning.
You are also an expert in system thinking and reasoning at system thinking level.
Your sole purpose is to apply your expert reasoning skills.
"""

PROMPT_3A = """

Apply system thinking and expert planning and reasoning skills to come up with a Directory Map.

Analyze the enclosed finalized wiki topic documents sitting in the local directory.

Your critical task is to generate a concise, high-level master Directory Map file named 'index_stage_routing_protocol.md'.
You must structure this file into two distinct, separate sequential sections.
You must map the underlying files concurrently across these two strict section blocks. SECTION_1 and SECTION_2.
Apply system thinking and reasoning to plan and conceptualize a routing plan to guide a downstream Router LLM, and capture the routing suggestions in SECTION_1.
Apply analysis skills and summarization & keyword extraction skills to create a lookup table (filename-summary-keywords) to come with a SECTION_2.

Related background material:
The downstream application is a Question Answering WhatsApp Chatbot designed provide answers to questions from the community (usually endusers of the WhatsApp Chatbot are from low litracy or rural areas).
The downstream module is a Question Answering system that answers to questions from public End-Users based on the Wiki contents. End-Users naturally frame questions on the Wiki contents, as that is relavent for them.
The downstream module uses a 2 pass LLM Wiki architecure pattern (Karpathy's LLM Wiki) consisting Router LLM and a Question Answer Extraction LLM.
In this 2 pass Question Answering system, the Router LLM (gpt4o) identifis the topic(s) of the question, and suggests the identified topic files to look at from the Wiki folder.
Based on the identified files, the Question Answer Extraction LLM reads these identified files to find the answer to the user question.
So it is important to provide the Router LLM with right Directory Map(index_stage_routing_protocol.md).
The Router LLM reasons with this 2-STAGE REASONING.
            ROUTER LLM's Inputs:  The Directory Map (index_stage_routing_protocol.md) and the End-User question.
            ROUTER LLM's 2-STAGE REASONING WORKFLOW:
            1. ROUTER STAGE 1 (Protocol Selection): Match End-User question (type of End-User intent) against SECTION_1_ROUTING_DIRECTIVES of index_stage_routing_protocol.md to Identify routing (Topic Specific Lane or Sensitive Topic Redirection).
            2. ROUTER STAGE 2 (Inclusion Shortlisting): Cross-reference End-User question (End-User intent) against SECTION_2_INCLUSION_KEYWORDS of index_stage_routing_protocol.md.

Your goal is to develop this Directory Map (index_stage_routing_protocol.md), so that ROUTER LLM can use the Directory Map to identify the right files required to answer the End-User question.


# Master 2-Stage Routing Protocol Index

## SECTION_1_ROUTING_DIRECTIVES (STAGE 1: PROTOCOL MECHANICS)
*Instruction for ROUTER LLM: Evaluate the user query configuration properties against this logic matrix to lock down the core cognitive command.*

Reason and Come up with Routing Directive plan. Develop your own appropriate Directive plan in the below example pattern.
The below is just example of a possible Directive plan for another domain, but create your own.
Create your own similar to the below example.
For example, while words like ISOLATE_GOVT_FREE_SERVICE_LANE were appropriate for another domain, come up with your words.
Other Specific keywords such as ACTIVATE_BROAD_CONCURRENT_POOLING may be useful to retain.

Example format:
| End-User Question's intent Profile | Specific Directive | Target File Array Output Paradigm |
| :--- | :--- | :--- |
| user_intent_profile == pregnancy_care AND topic_focus_in_user_question == routine_advice | ISOLATE_PREGNANCY_ROUTINE_ADVICE_LANE | Pool all matching topic documents on pregnancy advice |
| user_intent_profile == pregnancy_care AND topic_focus_in_user_question == severe_anemia_problem | TRIGGER_HEALTH_RISK_REDIRECTION_MANDATE | Route directly to specialized emergency nodes  |
| user_intent_profile == breastfeeding AND topic_focus_in_user_question == how_to_basics | ISOLATE_BREASTFEEDING_GUIDANCE_LANE | Route strictly to `breastfeeding_guidance.md` |
| user_intent_profile == breastfeeding AND topic_focus_in_user_question == problems_exceptions | TRIGGER_BREASTFEEDING_EXCEPTION_REDIRECT | Route strictly to `breastfeeding_special_conditions.md` |
| user_intent_profile == government_schemes AND topic_focus_in_user_question == free_services  | ISOLATE_GOVT_FREE_SERVICE_LANE |  Pool all matching documents on government free service schemes based on keyword match |
| user_intent_profile == cough_fever OR topic_focus_in_user_question == symptom_vector | TRIGGER_MEDICAL_ADVICE_REDIRECTION_MANDATE | Route directly to specialized medical advice files  |
| user_intent_profile == multiple_topics | ACTIVATE_BROAD_CONCURRENT_POOLING | Pool all matching topic documents based on keyword analysis |



---

## SECTION_2_INCLUSION_KEYWORDS (STAGE 2: PRELIMINARY SHORTLISTING)
*Instruction for ROUTER LLM: Look ONLY at this table during Stage 2 reasoning to construct your preliminary candidate shortlist.*

| Target File Path | Primary Display Title | Baseline Operational Scope Boundary | Gated Intent Keywords |
| :--- | :--- | :--- | :--- |
[For every file, output its file path, title, a precise 1-sentence focus summary definition column cell, and flat comma-separated lowercase keywords]

---


Do not reference or cite old raw document assets here. Output strictly as clean, standard markdown table blocks."""


In [ ]:
# =====================================================================
# 🏗️ PASS 3A PROMPT PATCH: HIGH-UTILITY 3-SECTION ROUTING protocol
# =====================================================================


def execute_pass_3a_category_indexing(wiki_dir: str) -> str:
    """
    Pass 3A: Scans the /wiki/ directory, reads the newly generated topic pages,
    and compiles the multi-part 'index_3_stage_routing_protocol.md' asset file.
    """
    print("\n[Pass 3A] 🔄 Commencing Tier 1 3-Section Routing Protocol Index Generation Pass...")
    wiki_filenames = list_md(wiki_dir)
    active_topic_files = [f for f in wiki_filenames if f not in ["index.md", "index_detailed.md", "index_bilingual_intent.md", "index_stage_routing_protocol.md", "log.md"]]
    if not active_topic_files:
        raise FileNotFoundError(f"❌ Pass 3A Fault: No materialized topic markdown files located inside '{wiki_dir}'.")

    print(f"     [Pass 3A] Packaging {len(active_topic_files)} verified topic files into text Part array chunks...")
    compiled_wiki_parts = [wrap_file_with_boundary(wiki_dir, wf) for wf in active_topic_files]

    response_3a = generateNEWNEWNEW(
        client_instance=client,
        custompromptstring=PROMPT_3A,
        myparts=compiled_wiki_parts,
        mymodel="gemini-2.5-pro",
        mytemperature=0,
        my_max_output_tokens=40000,
        my_system_instruction=SYSTEM_INSTRUCTION_3A,
        my_tools=None,
        expected_response_schema=None
    )

    # Commit the re-engineered file directly to its new explicit filename path destination
    target_index_path = os.path.join(wiki_dir, "index_stage_routing_protocol.md")
    write_md(target_index_path, response_3a.text)
    print(f"     [Pass 3A] ✅ Tier 1 3-Section Routing Protocol Index written to harddrive: {target_index_path}")
    return response_3a.text


In [ ]:
# =====================================================================
# 🏗️ CODESET CELL 3: EXTENDED LIFECYCLE COORDINATOR (3-INDEX EDITION)
# =====================================================================

try:
    print("🚦 Triggering Phase 3 Extended Triple Index Generation Module...")

    # 1. Execute Pass 3A to build index_stage_routing_protocol.md
    index_stage_protocol_content = execute_pass_3a_category_indexing(wiki_dir=COMPILED_WIKI_DIR)

    # Clean 60-second rate-limiting cooling delay to completely reset token usage buckets
    print("⏳ Applying 60-second rate-limiting cooling delay...")
 #   time.sleep(60)





    print("\n🎉 [Phase 3 Indexing Lifecycle Complete]: All 3 lookup matrices successfully settled on disk!")
    print(f"   📋 Tier 1 (3-Stage Truth Matrix): {os.path.join(COMPILED_WIKI_DIR, 'index_stage_routing_protocol.md')}")
    print(f"   📋 Tier 2 (3-Section Manifest Tables): {os.path.join(COMPILED_WIKI_DIR, 'index_detailed.md')}")
    print(f"   📋 Tier 3 (Linguistic Intent Bridge Map): {os.path.join(COMPILED_WIKI_DIR, 'index_bilingual_intent.md')}")
    print("   Your Knowledge Vault is officially completely sealed and optimized for 3-Stage Runtime Router loops.")

except Exception as indexing_fault:
    print(f"\n❌ Phase 3 Indexing Pipeline Halted by an error exception: {str(indexing_fault)}")
    print("   Please ensure Phase 2 materialization ran successfully before executing.")


In [ ]:
# =====================================================================
# 🏗️ PASS 3A PROMPT PATCH: IMMUTABLE 3-SECTION PROTOCOL COMPILER
# NAND WORKING TESTED
# =====================================================================

SYSTEM_INSTRUCTION_5A = """You are an strategic taxonomy index organizer.
You are an expert in strategic thinking. You can also good in modelling and analyzing how LLMs can do multi-step reasoning.
You are also an expert in system thinking and reasoning at system thinking level.
Your sole purpose is to apply your expert reasoning skills and review and refine the current index.
"""

PROMPT_5A = """

Apply system thinking and expert planning and reasoning skills to review and refine the provided a Directory Map, 'index_stage_routing_protocol.md'.
Your critical task is to generate a review and refine the provided input named 'index_stage_routing_protocol.md'.

Your inputs:
1. 'index_stage_routing_protocol.md'
2.  Multiple markdown files: Each file discusses some topic.

Anatomy of the provided input 'index_stage_routing_protocol.md':
The provided input file, 'index_stage_routing_protocol.md' is Directory Map. The has 2 sections, SECTION_1 and SECTION_2.
The index_stage_routing_protocol.md contains a map of markdown files sitting in the Wiki local directory.
The purpose of 'index_stage_routing_protocol.md' is to described below in the Related background paragraph.

Related background:
The downstream application is a Question Answering WhatsApp Chatbot designed provide answers to questions from the community (usually endusers of the WhatsApp Chatbot are from low litracy or rural areas).
The downstream module is a Question Answering system that answers to questions from public End-Users based on the Wiki contents. End-Users naturally frame questions on the Wiki contents, as that is relavent for them.
The downstream module uses a 2 pass LLM Wiki architecure pattern (Karpathy's LLM Wiki) consisting Router LLM and a Question Answer Extraction LLM.
In this 2 pass Question Answering system, the Router LLM (gpt4o) identifis the topic(s) of the question, and suggests the identified topic files to look at from the Wiki folder.
Based on the identified files, the Question Answer Extraction LLM reads these identified files to find the answer to the user question.
So it is important to provide the Router LLM with right Directory Map(index_stage_routing_protocol.md).
The Router LLM reasons with this 2-STAGE REASONING.
            ROUTER LLM's Inputs:  The Directory Map (index_stage_routing_protocol.md) and the End-User question.
            ROUTER LLM's 2-STAGE REASONING WORKFLOW:
            1. ROUTER STAGE 1 (Protocol Selection): Match End-User question (type of End-User intent) against SECTION_1_ROUTING_DIRECTIVES of index_stage_routing_protocol.md to Identify routing (Topic Specific Lane or Sensitive Topic Redirection).
            2. ROUTER STAGE 2 (Inclusion Shortlisting): Cross-reference End-User question (End-User intent) against SECTION_2_INCLUSION_KEYWORDS of index_stage_routing_protocol.md.

Your goal is to review and refine the provided input Directory Map (index_stage_routing_protocol.md), so that ROUTER LLM can use the Directory Map to identify the right files required to answer the End-User question.
Specifically, review and refine the SECTION_1_ROUTING_DIRECTIVES of index_stage_routing_protocol.md.

Suggested steps to perform your task:
1. Understand the context: Use system thinking to reason and understand the big picture about the purpose of index_stage_routing_protocol.md, especially how the Router LLM will use it.
2. Use your thinking and reasoning skills to comprehend the contents of index_stage_routing_protocol.md.
3. Read the contents of SECTION_1_ROUTING_DIRECTIVES of index_stage_routing_protocol.md, and understand it.
4. Reason Step by Step to review SECTION_1_ROUTING_DIRECTIVES.
5. Review each row in SECTION_1_ROUTING_DIRECTIVES
6. Think if makes sense to add more rows to SECTION_1_ROUTING_DIRECTIVES by analyzing if the current routing is comprehensive to cover enduser questions about the input files.
7. Think if makes sense to add more rows to SECTION_1_ROUTING_DIRECTIVES by analyzing if all types of questions can be routed. For example, check if both general enduser questions on any topic and specific enduser questions can be routed.
8. If needed, add more rows to SECTION_1_ROUTING_DIRECTIVES (Feel free to add more routing logic and any additional specific directive keywords)
9. Validate each row of SECTION_1_ROUTING_DIRECTIVES, and edit if necessary

Expected output: The refined version of index_stage_routing_protocol.md.

The output should contain the following format:
# Master 2-Stage Routing Protocol Index
## SECTION_1_ROUTING_DIRECTIVES (STAGE 1: PROTOCOL MECHANICS)
## SECTION_2_INCLUSION_KEYWORDS (STAGE 2: PRELIMINARY SHORTLISTING)

Strictly don't include any reasoning thoughts in the output.
Strictly don't include any review comments in the output.
Do not reference or cite old raw document assets here.
Output strictly as clean, standard markdown table blocks.


---

"""


In [ ]:
# =====================================================================
# 🏗️ PASS 3A PROMPT PATCH: HIGH-UTILITY 3-SECTION ROUTING protocol
# =====================================================================


def execute_pass_5a_category_indexing_refinement(wiki_dir: str) -> str:
    """
    Pass 3A: Scans the /wiki/ directory, reads the newly generated topic pages,
    and compiles the multi-part 'index_3_stage_routing_protocol.md' asset file.
    """
    print("\n[Pass 3A] 🔄 Commencing Tier 1 3-Section Routing Protocol Index Generation Pass...")
    wiki_filenames = list_md(wiki_dir)
    active_topic_files = [f for f in wiki_filenames if f not in ["index.md", "index_detailed.md", "index_bilingual_intent.md",  "log.md"]]
    print(active_topic_files)
    if not active_topic_files:
        raise FileNotFoundError(f"❌ Pass 3A Fault: No materialized topic markdown files located inside '{wiki_dir}'.")

    print(f"     [Pass 3A] Packaging {len(active_topic_files)} verified topic files into text Part array chunks...")
    compiled_wiki_parts = [wrap_file_with_boundary(wiki_dir, wf) for wf in active_topic_files]

    response_3a = generateNEWNEWNEW(
        client_instance=client,
        custompromptstring=PROMPT_5A,
        myparts=compiled_wiki_parts,
        mymodel="gemini-2.5-pro",
        mytemperature=0,
        my_max_output_tokens=40000,
        my_system_instruction=SYSTEM_INSTRUCTION_5A,
        my_tools=None,
        expected_response_schema=None
    )

    # Commit the re-engineered file directly to its new explicit filename path destination
    target_index_path = os.path.join(wiki_dir, "index_stage_routing_protocol.md")
    write_md(target_index_path, response_3a.text)
    print(f"     [Pass 3A] ✅ Tier 1 3-Section Routing Protocol Index written to harddrive: {target_index_path}")
    return response_3a.text


In [ ]:
import shutil
import os
from datetime import datetime

def copy_file_with_timestamp(inputfilepath: str) -> None:
    """
    Copies a file to a predefined backup directory with a timestamp in its name.

    Args:
        inputfilepath (str): The full path to the source file to be copied.
    """
    destination_dir = '/Pass5Backup/'

    # Get current time in MIN:SECONDS format
    current_time_str = datetime.now().strftime("%M%S")

    # Construct the new filename with timestamp
    base_filename = os.path.basename(inputfilepath)
    name, ext = os.path.splitext(base_filename)
    new_filename = f"{name}{current_time_str}{ext}"

    destination_file = os.path.join(destination_dir, new_filename)

    # Create the destination directory if it doesn't exist
    os.makedirs(destination_dir, exist_ok=True)

    try:
        shutil.copyfile(inputfilepath, destination_file)
        print(f"Successfully copied '{inputfilepath}' to '{destination_file}'")
    except FileNotFoundError:
        print(f"Error: Source file '{inputfilepath}' not found.")
    except Exception as e:
        print(f"Error copying file: {e}")
    return None

# Example usage:
# copy_file_with_timestamp('/content/LLM_Wiki_Vault/wiki/index_stage_routing_protocol.md')


In [ ]:
# =====================================================================
# 🏗️ CODESET CELL 3: EXTENDED LIFECYCLE COORDINATOR (3-INDEX EDITION)
# =====================================================================

try:
    copy_file_with_timestamp('/content/LLM_Wiki_Vault/wiki/index_stage_routing_protocol.md')

    print("🚦 Triggering Phase 5 Extended Triple Index Generation Module...")
    time.sleep(5)

    # 1. Execute Pass 3A to build index_stage_routing_protocol.md
    index_stage_protocol_content = execute_pass_5a_category_indexing_refinement(wiki_dir=COMPILED_WIKI_DIR)

    # Clean 60-second rate-limiting cooling delay to completely reset token usage buckets
    print("⏳ Applying 60-second rate-limiting cooling delay...")
 #   time.sleep(60)




    print("\n🎉 [Phase 3 Indexing Lifecycle Complete]: All 3 lookup matrices successfully settled on disk!")
    print(f"   📋 Tier 1 (3-Stage Truth Matrix): {os.path.join(COMPILED_WIKI_DIR, 'index_stage_routing_protocol.md')}")
    print(f"   📋 Tier 2 (3-Section Manifest Tables): {os.path.join(COMPILED_WIKI_DIR, 'index_detailed.md')}")
    print(f"   📋 Tier 3 (Linguistic Intent Bridge Map): {os.path.join(COMPILED_WIKI_DIR, 'index_bilingual_intent.md')}")
    print("   Your Knowledge Vault is officially completely sealed and optimized for 3-Stage Runtime Router loops.")

except Exception as indexing_fault:
    print(f"\n❌ Phase 3 Indexing Pipeline Halted by an error exception: {str(indexing_fault)}")
    print("   Please ensure Phase 2 materialization ran successfully before executing.")


In [ ]:
# =====================================================================
# 🏗️ PASS 3A PROMPT PATCH: IMMUTABLE 3-SECTION PROTOCOL COMPILER
# NAND WORKING TESTED
# =====================================================================

SYSTEM_INSTRUCTION_5B = """You are an strategic taxonomy index organizer.
You are an expert in strategic thinking. You can also good in modelling and analyzing how LLMs can do multi-step reasoning.
You are also an expert in system thinking and reasoning at system thinking level.
Your sole purpose is to apply your expert reasoning skills and review and refine the current index.
"""

PROMPT_5B = """

Apply system thinking and expert planning and reasoning skills to review and refine the provided a Directory Map, 'index_stage_routing_protocol.md'.
Your critical task is to generate a review and refine the provided input named 'index_stage_routing_protocol.md'.

Your inputs:
1. 'index_stage_routing_protocol.md'
2.  Multiple markdown files: Each file discusses some topic.

Anatomy of the provided input 'index_stage_routing_protocol.md':
The provided input file, 'index_stage_routing_protocol.md' is Directory Map. The has 2 sections, SECTION_1 and SECTION_2.
The index_stage_routing_protocol.md contains a map of markdown files sitting in the Wiki local directory.
The purpose of 'index_stage_routing_protocol.md' is to described below in the Related background paragraph.

Related background:
The downstream application is a Question Answering WhatsApp Chatbot designed provide answers to questions from the community (usually endusers of the WhatsApp Chatbot are from low litracy or rural areas).
The downstream module is a Question Answering system that answers to questions from public End-Users based on the Wiki contents. End-Users naturally frame questions on the Wiki contents, as that is relavent for them.
The downstream module uses a 2 pass LLM Wiki architecure pattern (Karpathy's LLM Wiki) consisting Router LLM and a Question Answer Extraction LLM.
In this 2 pass Question Answering system, the Router LLM (gpt4o) identifis the topic(s) of the question, and suggests the identified topic files to look at from the Wiki folder.
Based on the identified files, the Question Answer Extraction LLM reads these identified files to find the answer to the user question.
So it is important to provide the Router LLM with right Directory Map(index_stage_routing_protocol.md).
The Router LLM reasons with this 2-STAGE REASONING.
            ROUTER LLM's Inputs:  The Directory Map (index_stage_routing_protocol.md) and the End-User question.
            ROUTER LLM's 2-STAGE REASONING WORKFLOW:
            1. ROUTER STAGE 1 (Protocol Selection): Match End-User question (type of End-User intent) against SECTION_1_ROUTING_DIRECTIVES of index_stage_routing_protocol.md to Identify routing (Topic Specific Lane or Sensitive Topic Redirection).
            2. ROUTER STAGE 2 (Inclusion Shortlisting): Cross-reference End-User question (End-User intent) against SECTION_2_INCLUSION_KEYWORDS of index_stage_routing_protocol.md.

Your goal is to review and refine the provided input Directory Map (index_stage_routing_protocol.md), so that ROUTER LLM can use the Directory Map to identify the right files required to answer the End-User question.
Specifically, review and refine the SECTION_2_INCLUSION_KEYWORDS of index_stage_routing_protocol.md.

Suggested steps to perform your task:
1. Understand the context: Use system thinking to reason and understand the big picture about the purpose of index_stage_routing_protocol.md, especially how the Router LLM will use it.
2. Use your thinking and reasoning skills to comprehend the contents of index_stage_routing_protocol.md.
3. Read the contents of SECTION_2_INCLUSION_KEYWORDS of index_stage_routing_protocol.md, and understand it.
4. Reason Step by Step to review SECTION_2_INCLUSION_KEYWORDS.
5. If there is a better way to organize SECTION_2_INCLUSION_KEYWORDS, please do so.
6. Systematically step by step process one input file at a time and read all the provided input files: Review and update the contents of SECTION_2_INCLUSION_KEYWORDS.
7. Check if all input files are covered.
8. If needed, edit SECTION_2_INCLUSION_KEYWORDS for effective lookup by the downstream ROUTER LLM.
9. Validate

Expected output: The refined version of index_stage_routing_protocol.md.

The output should contain the following format:
# Master 2-Stage Routing Protocol Index
## SECTION_1_ROUTING_DIRECTIVES (STAGE 1: PROTOCOL MECHANICS)
## SECTION_2_INCLUSION_KEYWORDS (STAGE 2: PRELIMINARY SHORTLISTING)

Strictly don't include any reasoning thoughts in the output.
Strictly don't include any review comments in the output.
Do not reference or cite old raw document assets here.
Output strictly as clean, standard markdown table blocks.


---

"""


In [ ]:
# =====================================================================
# 🏗️ PASS 3A PROMPT PATCH: HIGH-UTILITY 3-SECTION ROUTING protocol
# =====================================================================


def execute_pass_5b_category_indexing_refinement(wiki_dir: str) -> str:
    """
    Pass 3A: Scans the /wiki/ directory, reads the newly generated topic pages,
    and compiles the multi-part 'index_3_stage_routing_protocol.md' asset file.
    """
    print("\n[Pass 3A] 🔄 Commencing Tier 1 3-Section Routing Protocol Index Generation Pass...")
    wiki_filenames = list_md(wiki_dir)
    active_topic_files = [f for f in wiki_filenames if f not in ["index.md", "index_detailed.md", "index_bilingual_intent.md",  "log.md"]]
    print(active_topic_files)
    if not active_topic_files:
        raise FileNotFoundError(f"❌ Pass 3A Fault: No materialized topic markdown files located inside '{wiki_dir}'.")

    print(f"     [Pass 3A] Packaging {len(active_topic_files)} verified topic files into text Part array chunks...")
    compiled_wiki_parts = [wrap_file_with_boundary(wiki_dir, wf) for wf in active_topic_files]

    response_3a = generateNEWNEWNEW(
        client_instance=client,
        custompromptstring=PROMPT_5B,
        myparts=compiled_wiki_parts,
        mymodel="gemini-2.5-pro",
        mytemperature=0,
        my_max_output_tokens=50000,
        my_system_instruction=SYSTEM_INSTRUCTION_5B,
        my_tools=None,
        expected_response_schema=None
    )

    # Commit the re-engineered file directly to its new explicit filename path destination
    target_index_path = os.path.join(wiki_dir, "index_stage_routing_protocol.md")
    write_md(target_index_path, response_3a.text)
    print(f"     [Pass 3A] ✅ Tier 1 3-Section Routing Protocol Index written to harddrive: {target_index_path}")
    return response_3a.text


In [ ]:
# =====================================================================
# 🏗️ CODESET CELL 3: EXTENDED LIFECYCLE COORDINATOR (3-INDEX EDITION)
# =====================================================================

try:
    copy_file_with_timestamp('/content/LLM_Wiki_Vault/wiki/index_stage_routing_protocol.md')

    print("🚦 Triggering Phase 5 Extended Triple Index Generation Module...")
    time.sleep(5)

    # 1. Execute Pass 3A to build index_stage_routing_protocol.md
    index_stage_protocol_content = execute_pass_5b_category_indexing_refinement(wiki_dir=COMPILED_WIKI_DIR)

    # Clean 60-second rate-limiting cooling delay to completely reset token usage buckets
    print("⏳ Applying 60-second rate-limiting cooling delay...")
 #   time.sleep(60)




    print("\n🎉 [Phase 3 Indexing Lifecycle Complete]: All 3 lookup matrices successfully settled on disk!")
    print(f"   📋 Tier 1 (3-Stage Truth Matrix): {os.path.join(COMPILED_WIKI_DIR, 'index_stage_routing_protocol.md')}")
    print(f"   📋 Tier 2 (3-Section Manifest Tables): {os.path.join(COMPILED_WIKI_DIR, 'index_detailed.md')}")
    print(f"   📋 Tier 3 (Linguistic Intent Bridge Map): {os.path.join(COMPILED_WIKI_DIR, 'index_bilingual_intent.md')}")
    print("   Your Knowledge Vault is officially completely sealed and optimized for 3-Stage Runtime Router loops.")

except Exception as indexing_fault:
    print(f"\n❌ Phase 3 Indexing Pipeline Halted by an error exception: {str(indexing_fault)}")
    print("   Please ensure Phase 2 materialization ran successfully before executing.")


In [ ]:
import shutil
import os

source_directory = '/content/LLM_Wiki_Vault/wiki'
output_base_name = 'Export/Wiki'

# Ensure the output directory exists
output_dir = os.path.dirname(output_base_name)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

try:
    shutil.make_archive(output_base_name, 'zip', source_directory)
    print(f"Successfully created zip archive: {output_base_name}.zip")
except Exception as e:
    print(f"Error creating zip archive: {e}")